In [ ]:
# Waveform and Firing Rate Clustering Analysis
# This notebook performs comprehensive clustering analysis combining waveform data with firing rate and bursting metrics

import pynapple as nap
from spikeinterface import load_sorting_analyzer
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import fnmatch
import matplotlib as mpl
import re
import os
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import umap
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded successfully!")


In [ ]:
nwb_paths = [
    Path("/data_store2/neuropixels/nwb/old/NP37_B1/NP37_B1.nwb"), # mut - 1
    Path("/data_store2/neuropixels/nwb/old/NP37_B2/NP37_B2.nwb"), # mut - 1
    Path("/data_store2/neuropixels/nwb/old/NP64_B1/NP64_B1.nwb"), # wt - 2
    Path("/data_store2/neuropixels/nwb/old/NP89_B1/NP89_B1.nwb"), # mut - 1 (throw out imec1)
    Path("/data_store2/neuropixels/nwb/old/NP130_B2/NP130_B2.nwb"), # wt - 1
    #Path("/data_store2/neuropixels/nwb/old/NP157_B2/NP157_B2.nwb"),
]   

manual_exclude_lists = [
    [16, 91], # #np37_b1.imec0
    [76, 77, 88], # #np37_b2.imec0
    [12, 14, 217, 224, 226, 227, 360, 361, 382], # #np64_b1.imec0
    [32, 33, 34], # #np64_b1.imec1
    [123, 254], # #np89_b1.imec0
    [59, 186], # #np130_b2.imec0
]

subj_list = [1, 2, 3, 3, 4, 5]
path_list = ['ast', 'ast', 'gbm', 'gbm', 'ast', 'gbm']
grade_list = [2, 2, 4, 4, 2, 4]
yield_list = [21, 25, 5, 30, 16, 21]
opercular_list = [0, 0, 0, 0, 0, 0]
region_list = ['pMTG', 'pMTG', 'SMG', 'SMG', 'parsTr', 'SMG']
age_list = [36, 36, 67, 67, 28, 69]
gender_list = [0, 0, 0, 0, 1, 0] # 0 is female

def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')

print(f"Loaded {len(nwb_paths)} NWB files")
print(len(subj_list))
print(len(grade_list))
print(len(path_list))
print(len(yield_list))
print(len(opercular_list))
print(len(region_list))
print(len(age_list))
print(len(gender_list))
print(len(manual_exclude_lists))


In [ ]:
# define all helpers needed for metrics for clustering 

# Bursting capacity metrics based on autocorrelograms
def calculate_autocorrelogram(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate autocorrelogram for spike times.
    
    Parameters:
    - spike_times: array of spike times in seconds
    - bin_size: bin size in seconds (default 1ms)
    - window_size: window size in seconds (default 100ms)
    
    Returns:
    - bins: time bins
    - autocorr: autocorrelogram values
    """
    if len(spike_times) < 2:
        return np.array([0]), np.array([0])
    
    # Create bins
    bins = np.arange(-window_size, window_size + bin_size, bin_size)
    
    # Calculate autocorrelogram
    autocorr = np.zeros(len(bins) - 1)
    
    for i, spike_time in enumerate(spike_times):
        # Find all other spikes
        other_spikes = np.concatenate([spike_times[:i], spike_times[i+1:]])
        
        # Calculate time differences
        time_diffs = other_spikes - spike_time
        
        # Bin the differences
        hist, _ = np.histogram(time_diffs, bins=bins)
        autocorr += hist
    
    # Normalize by number of spikes
    autocorr = autocorr / len(spike_times)
    
    return bins[:-1], autocorr

def calculate_bursting_metrics(spike_times, bin_size=0.001, window_size=0.1):
    """
    Calculate multiple bursting capacity metrics from autocorrelogram.
    
    Returns a dictionary with various bursting metrics:
    1. Burst Index: ratio of short-interval spikes to long-interval spikes
    2. Refractory Period Violation: spikes in refractory period
    3. Burst Peak Height: height of the first peak after time 0
    4. Burst Peak Width: width of the first peak
    5. Autocorr Skewness: skewness of the autocorrelogram
    6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    """
    if len(spike_times) < 10:  # Need sufficient spikes for reliable metrics
        return {
            'burst_index': 0,
            'refractory_violation': 0,
            'burst_peak_height': 0,
            'burst_peak_width': 0,
            'autocorr_skewness': 0,
            'short_isi_ratio': 0,
            'autocorr_vector': np.zeros(200)  # Fixed size for consistency
        }
    
    # Calculate autocorrelogram
    bins, autocorr = calculate_autocorrelogram(spike_times, bin_size, window_size)
    
    # Find center bin (time = 0)
    center_idx = len(bins) // 2
    
    # 1. Burst Index: ratio of short-interval spikes (1-10ms) to long-interval spikes (50-100ms)
    short_window = (bins >= 0.001) & (bins <= 0.010)  # 1-10ms
    long_window = (bins >= 0.050) & (bins <= 0.100)   # 50-100ms
    
    short_count = np.sum(autocorr[short_window])
    long_count = np.sum(autocorr[long_window])
    burst_index = short_count / (long_count + 1e-10)  # Add small value to avoid division by zero
    
    # 2. Refractory Period Violation: spikes in 0-2ms window
    refractory_window = (bins >= 0) & (bins <= 0.002)
    refractory_violation = np.sum(autocorr[refractory_window])
    
    # 3. Burst Peak Height: height of the first peak after time 0
    # Look for peaks in the 1-20ms window
    peak_window = (bins >= 0.001) & (bins <= 0.020)
    if np.any(peak_window):
        burst_peak_height = np.max(autocorr[peak_window])
    else:
        burst_peak_height = 0
    
    # 4. Burst Peak Width: width at half height of the first peak
    if burst_peak_height > 0:
        half_height = burst_peak_height / 2
        # Fix the indexing issue - use the correct window for peak_indices
        peak_indices = np.where(autocorr[peak_window] >= half_height)[0]
        if len(peak_indices) > 0:
            burst_peak_width = (np.max(peak_indices) - np.min(peak_indices)) * bin_size
        else:
            burst_peak_width = 0
    else:
        burst_peak_width = 0
    
    # 5. Autocorr Skewness: skewness of the autocorrelogram
    autocorr_skewness = stats.skew(autocorr)
    
    # 6. Short ISI Ratio: ratio of ISIs < 10ms to total ISIs
    isis = np.diff(spike_times)
    short_isis = np.sum(isis < 0.010)
    total_isis = len(isis)
    short_isi_ratio = short_isis / (total_isis + 1e-10)
    
    # Create a standardized autocorr vector (fixed size for consistency)
    autocorr_vector = np.zeros(200)
    if len(autocorr) > 0:
        # Interpolate to fixed size
        autocorr_vector = np.interp(np.linspace(0, len(autocorr)-1, 200), 
                                  np.arange(len(autocorr)), autocorr)
    
    return {
        'burst_index': burst_index,
        'refractory_violation': refractory_violation,
        'burst_peak_height': burst_peak_height,
        'burst_peak_width': burst_peak_width,
        'autocorr_skewness': autocorr_skewness,
        'short_isi_ratio': short_isi_ratio,
        'autocorr_vector': autocorr_vector
    }

# COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS
# Based on literature review for distinguishing interneurons vs pyramidal neurons

print("\n=== COMPREHENSIVE SPIKE CHARACTERISTICS ANALYSIS ===")
print("Selected metrics based on literature review:")
print("Waveform: Spike Width, Amplitude, Asymmetry, Rise Time, Decay Time")
print("Firing Rate: Mean Firing Rate, Burst Index, ISI CV, ISI Violation Rate, Spike Frequency Adaptation")

# Import additional libraries for comprehensive analysis
from sklearn.mixture import GaussianMixture
from scipy.stats import chi2_contingency, fisher_exact
import matplotlib.patches as mpatches

def calculate_spike_width(waveform):
    """Calculate spike width (trough-to-peak duration)"""
    # Find peak and trough
    peak_idx = np.argmax(waveform)
    trough_idx = np.argmin(waveform)
    
    # Calculate width in samples (assuming 30kHz sampling rate)
    width_samples = abs(peak_idx - trough_idx)
    width_ms = width_samples / 30.0  # Convert to milliseconds
    
    return width_ms

def calculate_spike_amplitude(waveform):
    """Calculate spike amplitude (peak-to-trough amplitude).
    Returns positive if abs(peak) > abs(trough), negative if abs(trough) > abs(peak)."""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    amplitude = peak_amp - trough_amp
    if abs(trough_amp) > abs(peak_amp):
        amplitude = -abs(amplitude)
    return amplitude

def calculate_spike_asymmetry(waveform):
    """Calculate spike asymmetry (peak/trough ratio)"""
    peak_amp = np.max(waveform)
    trough_amp = np.min(waveform)
    
    # Avoid division by zero
    if abs(trough_amp) < 1e-10:
        return np.inf
    
    asymmetry = abs(peak_amp / trough_amp)
    return asymmetry

def calculate_spike_rise_time(waveform):
    """Calculate spike rise time (baseline to peak)"""
    # Find baseline (first 10% of waveform)
    baseline_end = len(waveform) // 10
    baseline = np.mean(waveform[:baseline_end])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going up
    rising_phase = waveform[:peak_idx]
    baseline_crossings = np.where(np.diff(np.sign(rising_phase - baseline)) > 0)[0]
    
    if len(baseline_crossings) > 0:
        rise_start = baseline_crossings[-1]  # Last crossing before peak
        rise_time_samples = peak_idx - rise_start
        rise_time_ms = rise_time_samples / 30.0  # Convert to milliseconds
    else:
        rise_time_ms = peak_idx / 30.0  # Fallback
    
    return rise_time_ms

def calculate_spike_decay_time(waveform):
    """Calculate spike decay time (peak to baseline)"""
    # Find baseline (last 10% of waveform)
    baseline_start = int(len(waveform) * 0.9)
    baseline = np.mean(waveform[baseline_start:])
    
    # Find peak
    peak_idx = np.argmax(waveform)
    
    # Find where waveform crosses baseline going down after peak
    decay_phase = waveform[peak_idx:]
    baseline_crossings = np.where(np.diff(np.sign(decay_phase - baseline)) < 0)[0]
    
    if len(baseline_crossings) > 0:
        decay_end = baseline_crossings[0]  # First crossing after peak
        decay_time_samples = decay_end
        decay_time_ms = decay_time_samples / 30.0  # Convert to milliseconds
    else:
        decay_time_ms = (len(waveform) - peak_idx) / 30.0  # Fallback
    
    return decay_time_ms

def calculate_isi_cv(spike_times):
    """Calculate ISI coefficient of variation"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    if len(isis) == 0:
        return 0
    
    mean_isi = np.mean(isis)
    if mean_isi == 0:
        return 0
    
    cv = np.std(isis) / mean_isi
    return cv

def calculate_isi_violation_rate(spike_times, refractory_period=0.002):
    """Calculate ISI violation rate (spikes within refractory period)"""
    if len(spike_times) < 2:
        return 0
    
    isis = np.diff(spike_times)
    violations = np.sum(isis < refractory_period)
    total_spikes = len(spike_times)
    
    violation_rate = violations / total_spikes if total_spikes > 0 else 0
    return violation_rate

def calculate_spike_frequency_adaptation(spike_times, window_size=1.0):
    """Calculate spike frequency adaptation"""
    if len(spike_times) < 10:
        return 0
    
    # Divide spike train into windows
    total_time = spike_times[-1] - spike_times[0]
    n_windows = int(total_time / window_size)
    
    if n_windows < 2:
        return 0
    
    window_firing_rates = []
    for i in range(n_windows):
        window_start = spike_times[0] + i * window_size
        window_end = window_start + window_size
        
        spikes_in_window = np.sum((spike_times >= window_start) & (spike_times < window_end))
        firing_rate = spikes_in_window / window_size
        window_firing_rates.append(firing_rate)
    
    if len(window_firing_rates) < 2:
        return 0
    
    # Calculate adaptation as decrease in firing rate over time
    early_rate = np.mean(window_firing_rates[:len(window_firing_rates)//2])
    late_rate = np.mean(window_firing_rates[len(window_firing_rates)//2:])
    
    if early_rate == 0:
        return 0
    
    adaptation = (early_rate - late_rate) / early_rate
    return adaptation

print("Spike characteristic calculation functions defined!")

In [ ]:
# MAIN LOOP TO EXTRACT WAVEFORMS, FIRING RATES, AND COMPREHENSIVE METRICS
print("Starting data extraction from all sessions...")

# Initialize data storage lists
indicesAll = []
firingRatesAll = []
waveformAll = []
burstingMetricsAll = []
autocorrVectorsAll = []
spikeTimesAll = []
insertion = 0

for i in range(len(nwb_paths)):
    print(f"\nProcessing session {i+1}/{len(nwb_paths)}: {nwb_paths[i].name}")
    
    try:
        # Load NWB file
        data = nap.load_file(nwb_paths[i])
        keys = data.keys()

        # Filter keys similar to reference file
        template = "*imec*"
        keys = [key for key in keys if fnmatch.fnmatch(key, template)]
        ks_keys = [key for key in keys if "KS4" in key]
        if ks_keys:
            keys = ks_keys
        th8_keys = [key for key in keys if "Th=8" in key]
        if th8_keys:
            keys = th8_keys
        else:
            th_keys = [key for key in keys if "Th=" in key]
            if th_keys:
                keys = th_keys
        template_sentgen = "*sentgen*"
        template_auto = "*_auto*"
        keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
        # For NP137, remove 'imec1'
        if ('NP89' in str(nwb_paths[i])):
            keys = [key for key in keys if 'imec1' not in key]
        keys = sorted(keys, key=imec_key_sorter)

        for s in range(len(keys)):
            print(f"Processing {keys[s]}")
            
            spike_times = data[keys[s]]

            # Get firing rates and task times
            firingRates_all = spike_times.metadata["rate"]
            if "TaskTimes" in data.keys():
                task_times = data["TaskTimes"]
                beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)
                spike_times_beh = spike_times.restrict(beh_epochs)
                firingRates_beh = spike_times_beh.metadata["rate"]
                start_time = task_times.start
                end_time = task_times.end
            else:
                spike_times_beh = spike_times
                firingRates_beh = firingRates_all
                all_spike_times = np.concatenate([spike_times[u].as_series().index.values for u in spike_times])
                if all_spike_times.size > 0:
                    start_time = np.array([all_spike_times.min()])
                    end_time = np.array([all_spike_times.max()])
                else:
                    start_time = np.array([0.])
                    end_time = np.array([0.])

            # Quality checks (KS test, violation percentage)
            ks_stats = np.zeros(len(spike_times))
            ks_pvals = np.zeros(len(spike_times))
            for u in range(len(spike_times)):
                test = spike_times[u].as_series().index.values
                if len(test) > 1:
                    min_time = start_time[0]
                    max_time = end_time[-1]
                    normalized_spike_times = (test - min_time) / (max_time - min_time)
                    ks_result = kstest(normalized_spike_times, 'uniform')
                    ks_stats[u] = ks_result.statistic
                    ks_pvals[u] = ks_result.pvalue
                else:
                    ks_stats[u] = np.nan
                    ks_pvals[u] = np.nan

            violationThreshold = 3 / 1000
            violationPct = np.zeros(len(spike_times))
            for u in range(len(spike_times)):
                unit = spike_times[u]
                unit = unit.as_series().index
                if len(unit) < 100:
                    violationPct[u] = 1
                else:
                    isi = unit.diff()[1:]
                    violations = np.where(isi < violationThreshold)
                    violations = np.array(violations)
                    violationPct[u] = violations.size / len(isi)

            # Apply quality filters
            if "KSLabel" in spike_times.metadata:
                KSLabels = spike_times.metadata["KSLabel"]
            else:
                KSLabels = spike_times.metadata["quality"]
            firingRates = firingRates_beh
            mask1 = violationPct < 3 / 100
            mask2 = firingRates > 0.5
            mask3 = KSLabels != "noise"
            mask4 = ks_stats < 0.3
            mask = mask1 & mask2 & mask3 & mask4
            indicesFinal = firingRates.index[mask]

            # Exclude manually rejected neurons
            indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion])
            insertion += 1
            spike_times_good = spike_times[indicesFinal]
            print(f"Number of good neurons: {len(spike_times_good)}")

            if len(spike_times_good) == 0:
                print("No good neurons found, skipping session")
                continue

            # Get waveform and depth data
            kilosort_run = f'{keys[s]}_sparse'
            rec_id = os.path.basename(nwb_paths[i])  
            rec_id = os.path.splitext(rec_id)[0] 
            analyzer_path = f'/data_store2/neuropixels/nwb/old/{rec_id}/SI/{kilosort_run}'
            
            if not os.path.exists(analyzer_path):
                print(f"Analyzer path not found: {analyzer_path}")
                continue
                
            analyzer = load_sorting_analyzer(analyzer_path)
            templates = analyzer.get_extension("templates")
            avg_templates = templates.get_data(operator="average")

            waveform_list = []
            firing_rates_list = []
            bursting_metrics_list = []
            autocorr_vectors_list = []
            spike_times_list = []  # NEW: Store spike times for comprehensive analysis
            
            for unit_id in indicesFinal:
                # Get waveform
                template = avg_templates[unit_id]
                max_amp_per_channel = np.max(np.abs(template), axis=0)
                max_amp_ch = np.argmax(max_amp_per_channel)
                waveform = template[:, max_amp_ch]
                waveform_list.append(waveform)
                
                # Get firing rate
                firing_rate = firingRates.loc[unit_id]
                firing_rates_list.append(firing_rate)
                
                # Calculate bursting metrics
                spike_times_unit = spike_times_good[unit_id].as_series().index.values
                bursting_metrics = calculate_bursting_metrics(spike_times_unit)
                bursting_metrics_list.append(bursting_metrics)
                autocorr_vectors_list.append(bursting_metrics['autocorr_vector'])
                
                # NEW: Store spike times for comprehensive analysis
                spike_times_list.append(spike_times_unit)
            
            # Store data
            indicesAll.append(indicesFinal)
            firingRatesAll.append(firing_rates_list)
            waveformAll.append(waveform_list)
            burstingMetricsAll.append(bursting_metrics_list)
            autocorrVectorsAll.append(autocorr_vectors_list)
            spikeTimesAll.append(spike_times_list)  # NEW: Store spike times
            
    except Exception as e:
        print(f"Error processing session {i+1}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\nData extraction complete!")
print(f"Processed {insertion} sessions successfully")
print(f"Total waveforms: {sum(len(w) for w in waveformAll)}")
print(f"Total firing rate measurements: {sum(len(f) for f in firingRatesAll)}")
print(f"Total bursting metric measurements: {sum(len(b) for b in burstingMetricsAll)}")
print(f"Total spike time measurements: {sum(len(s) for s in spikeTimesAll)}")  # NEW


In [ ]:
# PREPARE DATA FOR CLUSTERING
print("\n=== PREPARING DATA FOR CLUSTERING ===")

# Flatten all data into single arrays
waveforms_flat = []
firing_rates_flat = []
bursting_metrics_flat = []
autocorr_vectors_flat = []
spike_times_flat = []  # NEW: Flatten spike times
metadata_flat = []

for insertion_idx in range(len(waveformAll)):
    for neuron_idx in range(len(waveformAll[insertion_idx])):
        waveforms_flat.append(waveformAll[insertion_idx][neuron_idx])
        firing_rates_flat.append(firingRatesAll[insertion_idx][neuron_idx])
        bursting_metrics_flat.append(burstingMetricsAll[insertion_idx][neuron_idx])
        autocorr_vectors_flat.append(autocorrVectorsAll[insertion_idx][neuron_idx])
        spike_times_flat.append(spikeTimesAll[insertion_idx][neuron_idx])  # NEW
        
        metadata_flat.append({
            'insertion_idx': insertion_idx,
            'subj': subj_list[insertion_idx],
            'grade': grade_list[insertion_idx],
            'pathology': path_list[insertion_idx],
            'yield': yield_list[insertion_idx],
            'opercular': opercular_list[insertion_idx],
            'region': region_list[insertion_idx],
            'age': age_list[insertion_idx],
            'gender': gender_list[insertion_idx],
        })

# Convert to numpy arrays
waveforms_array = np.array(waveforms_flat)
firing_rates_array = np.array(firing_rates_flat)
autocorr_vectors_array = np.array(autocorr_vectors_flat)

print(f"Total neurons: {len(waveforms_flat)}")
print(f"Waveform array shape: {waveforms_array.shape}")
print(f"Firing rates array shape: {firing_rates_array.shape}")
print(f"Autocorr vectors array shape: {autocorr_vectors_array.shape}")
print(f"Spike times data: {len(spike_times_flat)} neurons")  # NEW

# Extract individual bursting metrics
burst_indices = np.array([bm['burst_index'] for bm in bursting_metrics_flat])
refractory_violations = np.array([bm['refractory_violation'] for bm in bursting_metrics_flat])
burst_peak_heights = np.array([bm['burst_peak_height'] for bm in bursting_metrics_flat])
burst_peak_widths = np.array([bm['burst_peak_width'] for bm in bursting_metrics_flat])
autocorr_skewness = np.array([bm['autocorr_skewness'] for bm in bursting_metrics_flat])
short_isi_ratios = np.array([bm['short_isi_ratio'] for bm in bursting_metrics_flat])

print(f"\nBursting metrics extracted:")
print(f"Burst indices: mean={np.mean(burst_indices):.3f}, std={np.std(burst_indices):.3f}")
print(f"Refractory violations: mean={np.mean(refractory_violations):.3f}, std={np.std(refractory_violations):.3f}")
print(f"Burst peak heights: mean={np.mean(burst_peak_heights):.3f}, std={np.std(burst_peak_heights):.3f}")
print(f"Burst peak widths: mean={np.mean(burst_peak_widths):.3f}, std={np.std(burst_peak_widths):.3f}")
print(f"Autocorr skewness: mean={np.mean(autocorr_skewness):.3f}, std={np.std(autocorr_skewness):.3f}")
print(f"Short ISI ratios: mean={np.mean(short_isi_ratios):.3f}, std={np.std(short_isi_ratios):.3f}")

# Standardize waveforms
waveform_scaler = StandardScaler()
waveforms_scaled = waveform_scaler.fit_transform(waveforms_array)
print(f"\nWaveforms standardized: shape={waveforms_scaled.shape}")

# Prepare spiking data (firing rates + bursting metrics)
spiking_data = np.column_stack([
    firing_rates_array,
    burst_indices,
    refractory_violations,
    burst_peak_heights,
    burst_peak_widths,
    autocorr_skewness,
    short_isi_ratios
])

# Standardize spiking data
spiking_scaler = StandardScaler()
spiking_data_scaled = spiking_scaler.fit_transform(spiking_data)
print(f"Spiking data standardized: shape={spiking_data_scaled.shape}")

# Prepare combined data (waveforms + spiking metrics)
# First standardize each component separately, then combine
combined_data = np.column_stack([
    waveforms_scaled,
    spiking_data_scaled
])

print(f"Combined data prepared: shape={combined_data.shape}")
print("Data preparation complete!")


In [ ]:
# COLLECT COMPREHENSIVE SPIKE CHARACTERISTICS DATA (CORRECTED VERSION)
print("\n=== COLLECTING COMPREHENSIVE SPIKE CHARACTERISTICS ===")

# Initialize lists for comprehensive characteristics
comprehensive_waveform_metrics = []
comprehensive_firing_rate_metrics = []
comprehensive_metadata = []

print("Processing all neurons for comprehensive spike characteristics...")

# Process each neuron to extract comprehensive metrics
for neuron_idx in range(len(waveforms_flat)):
    if neuron_idx % 100 == 0:
        print(f"Processing neuron {neuron_idx+1}/{len(waveforms_flat)}")
    
    # Get waveform
    waveform = waveforms_flat[neuron_idx]
    
    # Calculate waveform characteristics
    spike_width = calculate_spike_width(waveform)
    spike_amplitude = calculate_spike_amplitude(waveform)
    spike_asymmetry = calculate_spike_asymmetry(waveform)
    spike_rise_time = calculate_spike_rise_time(waveform)
    spike_decay_time = calculate_spike_decay_time(waveform)
    
    waveform_metrics = [spike_width, spike_amplitude, spike_asymmetry, spike_rise_time, spike_decay_time]
    comprehensive_waveform_metrics.append(waveform_metrics)
    
    # Get firing rate characteristics
    firing_rate = firing_rates_flat[neuron_idx]
    burst_index = bursting_metrics_flat[neuron_idx]['burst_index']
    
    # Calculate additional firing rate characteristics using actual spike times
    spike_times_unit = spike_times_flat[neuron_idx]
    
    # Calculate ISI CV from actual spike times
    isi_cv = calculate_isi_cv(spike_times_unit)
    
    # Calculate ISI violation rate from actual spike times
    isi_violation_rate = calculate_isi_violation_rate(spike_times_unit)
    
    # Calculate spike frequency adaptation from actual spike times
    spike_frequency_adaptation = calculate_spike_frequency_adaptation(spike_times_unit)
    
    firing_rate_metrics = [firing_rate, burst_index, isi_cv, isi_violation_rate, spike_frequency_adaptation]
    comprehensive_firing_rate_metrics.append(firing_rate_metrics)
    
    # Store metadata
    comprehensive_metadata.append(metadata_flat[neuron_idx])

# Convert to numpy arrays
comprehensive_waveform_array = np.array(comprehensive_waveform_metrics)
comprehensive_firing_rate_array = np.array(comprehensive_firing_rate_metrics)

print(f"\nComprehensive metrics collected:")
print(f"Waveform metrics shape: {comprehensive_waveform_array.shape}")
print(f"Firing rate metrics shape: {comprehensive_firing_rate_array.shape}")

# Display summary statistics
waveform_metric_names = ['Spike Width (ms)', 'Spike Amplitude', 'Spike Asymmetry', 'Rise Time (ms)', 'Decay Time (ms)']
firing_rate_metric_names = ['Firing Rate (Hz)', 'Burst Index', 'ISI CV', 'ISI Violation Rate', 'Spike Frequency Adaptation']

print("\nWaveform Metrics Summary:")
for i, name in enumerate(waveform_metric_names):
    mean_val = np.mean(comprehensive_waveform_array[:, i])
    std_val = np.std(comprehensive_waveform_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nFiring Rate Metrics Summary:")
for i, name in enumerate(firing_rate_metric_names):
    mean_val = np.mean(comprehensive_firing_rate_array[:, i])
    std_val = np.std(comprehensive_firing_rate_array[:, i])
    print(f"  {name}: {mean_val:.3f} ± {std_val:.3f}")

print("\nComprehensive data collection complete!")


In [ ]:
# STANDARDIZE COMPREHENSIVE METRICS AND PREPARE FOR CLUSTERING
print("\n=== STANDARDIZING COMPREHENSIVE METRICS ===")

# Standardize waveform metrics
waveform_scaler_comprehensive = StandardScaler()
comprehensive_waveform_scaled = waveform_scaler_comprehensive.fit_transform(comprehensive_waveform_array)

# Standardize firing rate metrics
firing_rate_scaler_comprehensive = StandardScaler()
comprehensive_firing_rate_scaled = firing_rate_scaler_comprehensive.fit_transform(comprehensive_firing_rate_array)

# Create combined dataset
comprehensive_combined_data = np.column_stack([
    comprehensive_waveform_scaled,
    comprehensive_firing_rate_scaled
])

print(f"Standardized waveform metrics shape: {comprehensive_waveform_scaled.shape}")
print(f"Standardized firing rate metrics shape: {comprehensive_firing_rate_scaled.shape}")
print(f"Combined comprehensive data shape: {comprehensive_combined_data.shape}")

# Apply UMAP dimensionality reduction to comprehensive metrics
print("\n=== APPLYING UMAP TO COMPREHENSIVE METRICS ===")

# UMAP parameters
n_neighbors = 15
min_dist = 0.1
n_components = 2
random_state = 42

# UMAP on comprehensive waveform metrics
print("Applying UMAP to comprehensive waveform metrics...")
umap_waveform_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
waveform_comprehensive_umap = umap_waveform_comprehensive.fit_transform(comprehensive_waveform_scaled)

# UMAP on comprehensive firing rate metrics
print("Applying UMAP to comprehensive firing rate metrics...")
umap_firing_rate_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
firing_rate_comprehensive_umap = umap_firing_rate_comprehensive.fit_transform(comprehensive_firing_rate_scaled)

# UMAP on combined comprehensive data
print("Applying UMAP to combined comprehensive data...")
umap_combined_comprehensive = umap.UMAP(
    n_neighbors=n_neighbors,
    min_dist=min_dist,
    n_components=n_components,
    random_state=random_state,
    metric='euclidean'
)
combined_comprehensive_umap = umap_combined_comprehensive.fit_transform(comprehensive_combined_data)

print(f"Comprehensive waveform UMAP shape: {waveform_comprehensive_umap.shape}")
print(f"Comprehensive firing rate UMAP shape: {firing_rate_comprehensive_umap.shape}")
print(f"Comprehensive combined UMAP shape: {combined_comprehensive_umap.shape}")

print("UMAP dimensionality reduction complete!")


In [ ]:
# COMPREHENSIVE CLUSTERING ANALYSIS
print("\n=== COMPREHENSIVE CLUSTERING ANALYSIS ===")

# Method 1: UMAP + K-means Clustering
print("\n--- METHOD 1: UMAP + K-means Clustering ---")

def find_optimal_k_comprehensive(data, k_range, method_name):
    """Find optimal number of clusters using silhouette score"""
    silhouette_scores = []
    inertias = []
    
    print(f"Testing optimal k for {method_name}:")
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(data)
        
        silhouette_avg = silhouette_score(data, labels)
        inertia = kmeans.inertia_
        
        silhouette_scores.append(silhouette_avg)
        inertias.append(inertia)
        
        print(f"k={k}: Silhouette Score = {silhouette_avg:.3f}, Inertia = {inertia:.1f}")
    
    optimal_k = k_range[np.argmax(silhouette_scores)]
    print(f"Optimal number of clusters: {optimal_k}")
    
    return optimal_k, silhouette_scores, inertias

# Test different k values
k_range = range(2, 8)

# Find optimal k for each UMAP embedding
optimal_k_waveform_comp, silhouette_scores_waveform_comp, inertias_waveform_comp = find_optimal_k_comprehensive(
    waveform_comprehensive_umap, k_range, "comprehensive waveform UMAP"
)

optimal_k_firing_rate_comp, silhouette_scores_firing_rate_comp, inertias_firing_rate_comp = find_optimal_k_comprehensive(
    firing_rate_comprehensive_umap, k_range, "comprehensive firing rate UMAP"
)

optimal_k_combined_comp, silhouette_scores_combined_comp, inertias_combined_comp = find_optimal_k_comprehensive(
    combined_comprehensive_umap, k_range, "comprehensive combined UMAP"
)

# Perform final clustering with optimal k values (EXCEPT: force 3 clusters for combined UMAP)
print("\nPerforming final UMAP + K-means clustering...")

# Waveform clustering
kmeans_waveform_comp_final = KMeans(n_clusters=optimal_k_waveform_comp, random_state=42, n_init=10)
waveform_comp_labels = kmeans_waveform_comp_final.fit_predict(waveform_comprehensive_umap)
print(f"Comprehensive waveform clustering completed with {optimal_k_waveform_comp} clusters")

# Firing rate clustering
kmeans_firing_rate_comp_final = KMeans(n_clusters=optimal_k_firing_rate_comp, random_state=42, n_init=10)
firing_rate_comp_labels = kmeans_firing_rate_comp_final.fit_predict(firing_rate_comprehensive_umap)
print(f"Comprehensive firing rate clustering completed with {optimal_k_firing_rate_comp} clusters")

# Combined clustering (FORCE 3 CLUSTERS)
forced_n_clusters_combined = 3
kmeans_combined_comp_final = KMeans(n_clusters=forced_n_clusters_combined, random_state=42, n_init=10)
combined_comp_labels = kmeans_combined_comp_final.fit_predict(combined_comprehensive_umap)
print(f"Comprehensive combined clustering completed with {forced_n_clusters_combined} clusters (forced, regardless of optimal)")

print("UMAP + K-means clustering complete!")


In [ ]:
# VISUALIZE UMAP CLUSTERS FORCED TO 3 CLUSTERS (COMBINED ONLY)
print("\n=== UMAP CLUSTER VISUALIZATION: Combined UMAP (3 Clusters Forced) ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

fig, ax = plt.subplots(figsize=(8, 7))
for cluster_id in range(3):
    mask = combined_comp_labels == cluster_id
    ax.plot(
        combined_comprehensive_umap[mask, 0],
        combined_comprehensive_umap[mask, 1],
        'o', alpha=0.7, markersize=5, markeredgewidth=0, markeredgecolor='none',
        color=plt.cm.tab10(cluster_id),
        label=f'Cluster {cluster_id}'
    )
ax.set_title('Combined UMAP - K-means Clusters (k=3, Forced)', fontsize=15, fontweight='bold')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.grid(True, alpha=0.3)
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig("combined_umap_clusters_validation.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

print("\n=== DETAILED CLUSTER ANALYSIS: Combined UMAP (3 Clusters Forced) ===")
def analyze_combined_cluster_characteristics_3(labels, waveform_data, firing_rate_data):
    for cluster_id in range(3):
        cluster_mask = labels == cluster_id
        cluster_size = np.sum(cluster_mask)
        if cluster_size > 0:
            cluster_waveform_metrics = waveform_data[cluster_mask]
            cluster_firing_rate_metrics = firing_rate_data[cluster_mask]
            print(f"  Cluster {cluster_id} (n={cluster_size}):")
            print(f"    Waveform Metrics:")
            print(f"      Spike Width:        {np.mean(cluster_waveform_metrics[:, 0]):.3f} ± {np.std(cluster_waveform_metrics[:, 0]):.3f} ms")
            print(f"      Spike Amplitude:    {np.mean(cluster_waveform_metrics[:, 1]):.3f} ± {np.std(cluster_waveform_metrics[:, 1]):.3f}")
            print(f"      Spike Asymmetry:    {np.mean(cluster_waveform_metrics[:, 2]):.3f} ± {np.std(cluster_waveform_metrics[:, 2]):.3f}")
            print(f"      Rise Time:          {np.mean(cluster_waveform_metrics[:, 3]):.3f} ± {np.std(cluster_waveform_metrics[:, 3]):.3f} ms")
            print(f"      Decay Time:         {np.mean(cluster_waveform_metrics[:, 4]):.3f} ± {np.std(cluster_waveform_metrics[:, 4]):.3f} ms")
            print(f"    Firing Rate Metrics:")
            print(f"      Firing Rate:            {np.mean(cluster_firing_rate_metrics[:, 0]):.2f} ± {np.std(cluster_firing_rate_metrics[:, 0]):.2f} Hz")
            print(f"      Burst Index:            {np.mean(cluster_firing_rate_metrics[:, 1]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 1]):.3f}")
            print(f"      ISI CV:                 {np.mean(cluster_firing_rate_metrics[:, 2]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 2]):.3f}")
            print(f"      ISI Violation Rate:     {np.mean(cluster_firing_rate_metrics[:, 3]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 3]):.3f}")
            print(f"      Spike Frequency Adapt.: {np.mean(cluster_firing_rate_metrics[:, 4]):.3f} ± {np.std(cluster_firing_rate_metrics[:, 4]):.3f}")

analyze_combined_cluster_characteristics_3(
    combined_comp_labels, comprehensive_waveform_array, comprehensive_firing_rate_array
)

print("\nCombined UMAP 3-cluster visualization complete!")

In [ ]:
# BAR PLOTS FOR ALL VARIABLES BY CLUSTER WITH STATISTICAL TESTS
print("\n=== VARIABLE ANALYSIS BY CLUSTER ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu

# Define variable names
waveform_vars = ['Spike Width (ms)', 'Spike Amplitude', 'Spike Asymmetry', 'Rise Time (ms)', 'Decay Time (ms)']
firing_rate_vars = ['Firing Rate (Hz)', 'Burst Index', 'ISI CV', 'ISI Violation Rate', 'Spike Frequency Adaptation']
all_vars = waveform_vars + firing_rate_vars

# Get the data arrays
waveform_data = comprehensive_waveform_array  # Shape: (n_neurons, 5)
firing_rate_data = comprehensive_firing_rate_array  # Shape: (n_neurons, 5)
combined_data = np.hstack([waveform_data, firing_rate_data])  # Shape: (n_neurons, 10)

# Get cluster labels
cluster_labels = combined_comp_labels

# Create figure with subplots for all variables
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Variable Analysis by Cluster (Combined UMAP)', fontsize=16, fontweight='bold')

# Flatten axes for easier indexing
axes_flat = axes.flatten()

# Function to perform statistical tests
def perform_statistical_tests(data_cluster0, data_cluster2, test_name):
    """Perform t-test and Mann-Whitney U test"""
    if len(data_cluster0) == 0 or len(data_cluster2) == 0:
        return None, None

    # T-test
    t_stat, t_p = ttest_ind(data_cluster0, data_cluster2)

    # Mann-Whitney U test
    u_stat, u_p = mannwhitneyu(data_cluster0, data_cluster2, alternative='two-sided')

    return {'t_test': (t_stat, t_p), 'mannwhitney': (u_stat, u_p)}

# Plot each variable
for var_idx in range(10):
    ax = axes_flat[var_idx]

    # Get data for this variable
    var_data = combined_data[:, var_idx]

    # Calculate means and standard errors for each cluster
    cluster_means = []
    cluster_stds = []
    cluster_sizes = []
    cluster_data = []

    for cluster_id in range(3):  # Assuming 3 clusters
        cluster_mask = cluster_labels == cluster_id
        cluster_var_data = var_data[cluster_mask]

        if len(cluster_var_data) > 0:
            cluster_means.append(np.mean(cluster_var_data))
            cluster_stds.append(np.std(cluster_var_data))
            cluster_sizes.append(len(cluster_var_data))
            cluster_data.append(cluster_var_data)
        else:
            cluster_means.append(0)
            cluster_stds.append(0)
            cluster_sizes.append(0)
            cluster_data.append(np.array([]))

    # Create bar plot
    x_pos = np.arange(3)
    bars = ax.bar(x_pos, cluster_means, yerr=cluster_stds, capsize=5,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7, edgecolor='black')

    # Add value labels on bars
    for i, (bar, mean, std, size) in enumerate(zip(bars, cluster_means, cluster_stds, cluster_sizes)):
        if size > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                   f'{mean:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=8)

    # Perform statistical tests between cluster 0 and cluster 2
    if len(cluster_data[0]) > 0 and len(cluster_data[2]) > 0:
        stats_results = perform_statistical_tests(cluster_data[0], cluster_data[2], all_vars[var_idx])

        if stats_results:
            t_stat, t_p = stats_results['t_test']
            u_stat, u_p = stats_results['mannwhitney']

            # Add significance annotation
            significance = "***" if t_p < 0.001 else "**" if t_p < 0.01 else "*" if t_p < 0.05 else "ns"
            ax.text(0.5, 0.95, f"Cluster 0 vs 2:\nT-test p = {t_p:.4f} {significance}",
                   transform=ax.transAxes, ha='center', va='top', fontweight='bold', fontsize=8,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    # Set plot properties
    ax.set_title(all_vars[var_idx], fontsize=10, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Value')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Cluster 0', 'Cluster 1', 'Cluster 2'])
    ax.grid(axis='y', alpha=0.3)

    # Print detailed statistics
    print(f"\n{all_vars[var_idx]}:")
    for cluster_id in range(3):
        if cluster_sizes[cluster_id] > 0:
            print(f"  Cluster {cluster_id}: mean = {cluster_means[cluster_id]:.3f} ± {cluster_stds[cluster_id]:.3f} (n={cluster_sizes[cluster_id]})")

    # Print statistical test results
    if len(cluster_data[0]) > 0 and len(cluster_data[2]) > 0:
        print(f"  Cluster 0 vs 2: T-test p = {t_p:.4f}, Mann-Whitney p = {u_p:.4f}")

plt.tight_layout()
plt.savefig("summary_stats_clusters_validation.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Summary statistics table
print("\n=== SUMMARY STATISTICS TABLE ===")
print("Variable\t\t\tCluster 0\t\tCluster 1\t\tCluster 2")
print("-" * 80)

for var_idx in range(10):
    var_name = all_vars[var_idx]
    # Pad variable name to align columns
    padded_name = var_name.ljust(25)

    # Get cluster data
    cluster_data = []
    for cluster_id in range(3):
        cluster_mask = cluster_labels == cluster_id
        cluster_var_data = combined_data[cluster_mask, var_idx]
        cluster_data.append(cluster_var_data)

    # Format means and standard deviations
    cluster_stats = []
    for cluster_id in range(3):
        if len(cluster_data[cluster_id]) > 0:
            mean_val = np.mean(cluster_data[cluster_id])
            std_val = np.std(cluster_data[cluster_id])
            cluster_stats.append(f"{mean_val:.3f} ± {std_val:.3f}")
        else:
            cluster_stats.append("N/A")

    print(f"{padded_name}\t{cluster_stats[0]}\t\t{cluster_stats[1]}\t\t{cluster_stats[2]}")

# Statistical significance summary
print("\n=== STATISTICAL SIGNIFICANCE SUMMARY (Cluster 0 vs 2) ===")
print("Variable\t\t\tT-test p-value\t\tMann-Whitney p-value")
print("-" * 70)

for var_idx in range(10):
    var_name = all_vars[var_idx]
    padded_name = var_name.ljust(25)

    # Get cluster data
    cluster_mask_0 = cluster_labels == 0
    cluster_mask_2 = cluster_labels == 2
    cluster_data_0 = combined_data[cluster_mask_0, var_idx]
    cluster_data_2 = combined_data[cluster_mask_2, var_idx]

    if len(cluster_data_0) > 0 and len(cluster_data_2) > 0:
        # T-test
        t_stat, t_p = ttest_ind(cluster_data_0, cluster_data_2)

        # Mann-Whitney U test
        u_stat, u_p = mannwhitneyu(cluster_data_0, cluster_data_2, alternative='two-sided')

        print(f"{padded_name}\t{t_p:.4f}\t\t\t{u_p:.4f}")
    else:
        print(f"{padded_name}\tN/A\t\t\tN/A")

print("\nVariable analysis complete!")

In [ ]:
# COMPREHENSIVE VISUALIZATION FUNCTIONS
print("\n=== COMPREHENSIVE VISUALIZATION FUNCTIONS (FORCING 3 CLUSTERS) ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

def plot_cluster_waveforms_comprehensive(labels, waveforms_array, title_prefix, n_clusters, method_name):
    """Plot average waveforms for each cluster (now always 3 clusters)"""
    n_clusters_fixed = 3
    fig, axes = plt.subplots(1, n_clusters_fixed, figsize=(4*n_clusters_fixed, 4))
    if n_clusters_fixed == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Average Waveforms ({method_name})', fontsize=14, fontweight='bold')
    
    for cluster_id in range(n_clusters_fixed):
        cluster_mask = labels == cluster_id
        cluster_waveforms = waveforms_array[cluster_mask]
        
        if len(cluster_waveforms) > 0:
            # Calculate average waveform
            avg_waveform = np.mean(cluster_waveforms, axis=0)
            std_waveform = np.std(cluster_waveforms, axis=0)
            
            # Plot average waveform with error bars
            x_vals = np.arange(len(avg_waveform)) / 30.0  # Convert to ms
            axes[cluster_id].plot(x_vals, avg_waveform, 'b-', linewidth=2, label=f'Cluster {cluster_id}')
            axes[cluster_id].fill_between(x_vals, 
                                        avg_waveform - std_waveform, 
                                        avg_waveform + std_waveform, 
                                        alpha=0.3, color='blue')
            
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_waveforms)})')
            axes[cluster_id].set_xlabel('Time (ms)')
            axes[cluster_id].set_ylabel('Amplitude')
            axes[cluster_id].grid(True, alpha=0.3)
            axes[cluster_id].legend()
    
    plt.tight_layout()
    plt.savefig("cluster_waveforms_validation.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

def plot_region_pie_charts(labels, metadata, title_prefix, n_clusters, method_name):
    """Plot pie charts showing regional distribution for each cluster (forcing 3 clusters)"""
    n_clusters_fixed = 3
    fig, axes = plt.subplots(1, n_clusters_fixed, figsize=(5*n_clusters_fixed, 4))
    if n_clusters_fixed == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Regional Distribution ({method_name})', fontsize=14, fontweight='bold')
    
    # Get unique regions
    all_regions = [meta['region'] for meta in metadata]
    unique_regions = list(set(all_regions))
    
    for cluster_id in range(n_clusters_fixed):
        cluster_mask = labels == cluster_id
        cluster_metadata = [metadata[i] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_metadata) > 0:
            # Count regions in this cluster
            cluster_regions = [meta['region'] for meta in cluster_metadata]
            region_counts = {region: cluster_regions.count(region) for region in unique_regions}
            
            # Create pie chart
            labels_pie = list(region_counts.keys())
            sizes = list(region_counts.values())
            colors = plt.cm.Set3(np.linspace(0, 1, len(labels_pie)))
            
            axes[cluster_id].pie(sizes, labels=labels_pie, autopct='%1.1f%%', colors=colors)
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_metadata)})')
    
    plt.tight_layout()
    plt.savefig("cluster_region_validation.svg", format="svg", dpi=300, bbox_inches='tight')
    plt.show()

def plot_depth_histograms(labels, metadata, title_prefix, n_clusters, method_name):
    """Plot depth histograms for each cluster (forcing 3 clusters)"""
    n_clusters_fixed = 3
    fig, axes = plt.subplots(1, n_clusters_fixed, figsize=(5*n_clusters_fixed, 4))
    if n_clusters_fixed == 1:
        axes = [axes]
    
    fig.suptitle(f'{title_prefix} - Depth Distribution ({method_name})', fontsize=14, fontweight='bold')
    
    for cluster_id in range(n_clusters_fixed):
        cluster_mask = labels == cluster_id
        cluster_depths = [metadata[i]['depth'] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_depths) > 0:
            axes[cluster_id].hist(cluster_depths, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            axes[cluster_id].set_title(f'Cluster {cluster_id} (n={len(cluster_depths)})')
            axes[cluster_id].set_xlabel('Depth')
            axes[cluster_id].set_ylabel('Count')
            axes[cluster_id].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_subgroup_bar_charts(labels, metadata, title_prefix, n_clusters, method_name):
    """Plot bar charts showing subgroup proportions for each cluster (forcing 3 clusters)"""
    n_clusters_fixed = 3
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    fig.suptitle(f'{title_prefix} - Subgroup Proportions ({method_name})', fontsize=14, fontweight='bold')
    
    # Subplot 1: Pathology (oli+ast vs gbm)
    ax1 = axes[0]
    pathology_data = []
    cluster_labels = []
    
    for cluster_id in range(n_clusters_fixed):
        cluster_mask = labels == cluster_id
        cluster_metadata = [metadata[i] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_metadata) > 0:
            # Count pathology types
            oli_ast_count = sum(1 for meta in cluster_metadata if meta['pathology'] in ['oli', 'ast'])
            gbm_count = sum(1 for meta in cluster_metadata if meta['pathology'] == 'gbm')
            total = len(cluster_metadata)
            
            # Calculate proportions
            oli_ast_prop = oli_ast_count / total if total > 0 else 0
            gbm_prop = gbm_count / total if total > 0 else 0
            
            pathology_data.append([oli_ast_prop, gbm_prop])
            cluster_labels.append(f'Cluster {cluster_id}')
        else:
            pathology_data.append([0, 0])
            cluster_labels.append(f'Cluster {cluster_id}')
    
    pathology_data = np.array(pathology_data)
    x = np.arange(len(cluster_labels))
    width = 0.35
    
    ax1.bar(x - width/2, pathology_data[:, 0], width, label='Oli + Ast', color='lightblue', alpha=0.8)
    ax1.bar(x + width/2, pathology_data[:, 1], width, label='GBM', color='lightcoral', alpha=0.8)
    
    ax1.set_xlabel('Cluster')
    ax1.set_ylabel('Proportion')
    ax1.set_title('Pathology Distribution (Oli+Ast vs GBM)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(cluster_labels)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Subplot 2: Grade (2+3 vs 4)
    ax2 = axes[1]
    grade_data = []
    
    for cluster_id in range(n_clusters_fixed):
        cluster_mask = labels == cluster_id
        cluster_metadata = [metadata[i] for i in range(len(metadata)) if cluster_mask[i]]
        
        if len(cluster_metadata) > 0:
            # Count grades
            grade_2_3_count = sum(1 for meta in cluster_metadata if meta['grade'] in [2, 3])
            grade_4_count = sum(1 for meta in cluster_metadata if meta['grade'] == 4)
            total = len(cluster_metadata)
            
            # Calculate proportions
            grade_2_3_prop = grade_2_3_count / total if total > 0 else 0
            grade_4_prop = grade_4_count / total if total > 0 else 0
            
            grade_data.append([grade_2_3_prop, grade_4_prop])
        else:
            grade_data.append([0, 0])
    
    grade_data = np.array(grade_data)
    
    ax2.bar(x - width/2, grade_data[:, 0], width, label='Grade 2+3', color='lightgreen', alpha=0.8)
    ax2.bar(x + width/2, grade_data[:, 1], width, label='Grade 4', color='orange', alpha=0.8)
    
    ax2.set_xlabel('Cluster')
    ax2.set_ylabel('Proportion')
    ax2.set_title('Grade Distribution (Grade 2+3 vs Grade 4)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(cluster_labels)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("Comprehensive visualization functions defined! (using 3 clusters for display)")


In [ ]:
# COMPREHENSIVE VISUALIZATIONS FOR UMAP + K-MEANS CLUSTERING
print("\n=== COMPREHENSIVE VISUALIZATIONS: UMAP + K-MEANS ===")
plt.rcParams['svg.fonttype'] = 'none'  # Ensures text is editable in Illustrator

# Convert waveforms_flat to numpy array for visualization
waveforms_array_for_viz = np.array(waveforms_flat)

# # 1. Waveform UMAP + K-means visualizations
# print("\n--- Waveform UMAP + K-means Clustering ---")
# plot_cluster_waveforms_comprehensive(waveform_comp_labels, waveforms_array_for_viz, 
#                                     "Waveform UMAP", optimal_k_waveform_comp, "UMAP+K-means")
# plot_region_pie_charts(waveform_comp_labels, comprehensive_metadata, 
#                       "Waveform UMAP", optimal_k_waveform_comp, "UMAP+K-means")
# plot_depth_histograms(waveform_comp_labels, comprehensive_metadata, 
#                      "Waveform UMAP", optimal_k_waveform_comp, "UMAP+K-means")
# plot_subgroup_bar_charts(waveform_comp_labels, comprehensive_metadata, 
#                         "Waveform UMAP", optimal_k_waveform_comp, "UMAP+K-means")

# # 2. Firing Rate UMAP + K-means visualizations
# print("\n--- Firing Rate UMAP + K-means Clustering ---")
# plot_cluster_waveforms_comprehensive(firing_rate_comp_labels, waveforms_array_for_viz, 
#                                     "Firing Rate UMAP", optimal_k_firing_rate_comp, "UMAP+K-means")
# plot_region_pie_charts(firing_rate_comp_labels, comprehensive_metadata, 
#                       "Firing Rate UMAP", optimal_k_firing_rate_comp, "UMAP+K-means")
# plot_depth_histograms(firing_rate_comp_labels, comprehensive_metadata, 
#                      "Firing Rate UMAP", optimal_k_firing_rate_comp, "UMAP+K-means")
# plot_subgroup_bar_charts(firing_rate_comp_labels, comprehensive_metadata, 
#                         "Firing Rate UMAP", optimal_k_firing_rate_comp, "UMAP+K-means")

# 3. Combined UMAP + K-means visualizations
print("\n--- Combined UMAP + K-means Clustering ---")
plot_cluster_waveforms_comprehensive(combined_comp_labels, waveforms_array_for_viz, 
                                    "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
plot_region_pie_charts(combined_comp_labels, comprehensive_metadata, 
                      "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_depth_histograms(combined_comp_labels, comprehensive_metadata, 
#                     "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")
#plot_subgroup_bar_charts(combined_comp_labels, comprehensive_metadata, 
#                        "Combined UMAP", optimal_k_combined_comp, "UMAP+K-means")

print("UMAP + K-means visualizations complete!")


In [ ]:
# LOOP THROUGH ALL INSERTIONS FOR E/I BALANCE ANALYSIS
# WITH BOTH ASSEMBLY AND NEURON INFORMATION CAPACITY ANALYSIS
import fnmatch
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import re
import os
from scipy import stats
from sklearn.decomposition import PCA, FastICA
from scipy.linalg import norm
from scipy.stats import kstest

def imec_key_sorter(key):
    """Extract the number after 'imec', default to a large number if not found."""
    match = re.search(r'imec(\d+)', key)
    return int(match.group(1)) if match else float('inf')

def calculate_mutual_information(x, y, bins=20):
    """
    Calculate mutual information between two variables using histogram-based approach.
    """
    valid_mask = ~(np.isnan(x) | np.isnan(y))
    if np.sum(valid_mask) < 2:
        return np.nan

    x_valid = x[valid_mask]
    y_valid = y[valid_mask]

    hist_2d, _, _ = np.histogram2d(x_valid, y_valid, bins=bins)
    hist_x = np.sum(hist_2d, axis=1)
    hist_y = np.sum(hist_2d, axis=0)

    p_xy = hist_2d / np.sum(hist_2d)
    p_x = hist_x / np.sum(hist_x)
    p_y = hist_y / np.sum(hist_y)

    mi = 0.0
    for i in range(len(p_x)):
        for j in range(len(p_y)):
            if p_xy[i, j] > 0 and p_x[i] > 0 and p_y[j] > 0:
                mi += p_xy[i, j] * np.log2(p_xy[i, j] / (p_x[i] * p_y[j]))
    return mi

def calculate_information_capacity(data, time_bins, min_samples=10):
    """
    Calculate information storage capacity using multiple metrics.
    """
    valid_mask = ~np.isnan(data)
    if np.sum(valid_mask) < min_samples:
        return {
            'entropy': np.nan,
            'mutual_info_temporal': np.nan,
            'capacity': np.nan,
            'n_valid_samples': np.sum(valid_mask),
            'data_range': np.nan,
            'data_std': np.nan
        }

    valid_data = data[valid_mask]

    n_bins = min(20, len(valid_data) // 5)
    if n_bins < 5:
        n_bins = 5

    hist, _ = np.histogram(valid_data, bins=n_bins)
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]

    if len(prob) < 2:
        entropy_val = np.nan
    else:
        entropy_val = -np.sum(prob * np.log2(prob))

    if len(valid_data) > 1:
        try:
            mi_temporal = calculate_mutual_information(
                valid_data[:-1],
                valid_data[1:],
                bins=min(10, len(valid_data) // 10)
            )
        except Exception:
            mi_temporal = np.nan
    else:
        mi_temporal = np.nan

    if not np.isnan(entropy_val) and not np.isnan(mi_temporal):
        capacity = entropy_val - mi_temporal
    elif not np.isnan(entropy_val):
        capacity = entropy_val
    else:
        capacity = np.nan

    data_range = np.max(valid_data) - np.min(valid_data) if len(valid_data) > 0 else np.nan
    data_std = np.std(valid_data) if len(valid_data) > 0 else np.nan

    return {
        'entropy': entropy_val,
        'mutual_info_temporal': mi_temporal,
        'capacity': capacity,
        'n_valid_samples': np.sum(valid_mask),
        'data_range': data_range,
        'data_std': data_std
    }

def calculate_expression_strength(firingRateMatrix, ap_norm_matrix):
    """
    Calculate expression strength using the method from the paper:
    E(b) = R(b)^T * Oi * R(b)
    """
    n_time_bins, n_neurons = firingRateMatrix.shape
    n_assemblies = ap_norm_matrix.shape[0]
    
    expression_strength = np.zeros((n_time_bins, n_assemblies))
    
    for assembly_idx in range(n_assemblies):
        weight_vector = ap_norm_matrix[assembly_idx, :]
        outer_product = np.outer(weight_vector, weight_vector)
        
        for time_bin in range(n_time_bins):
            firing_rate_vector = firingRateMatrix[time_bin, :]
            expression_strength[time_bin, assembly_idx] = firing_rate_vector.T @ outer_product @ firing_rate_vector
    
    return expression_strength

def calculate_ei_balance(assembly_pattern, cluster_labels):
    """
    Calculate E/I balance for an assembly pattern.
    Multiply by -1 for cluster 1, 0 for cluster 2, +1 for cluster 0.
    """
    ei_balance = 0.0
    for neuron_idx, weight in enumerate(assembly_pattern):
        cluster_id = cluster_labels[neuron_idx]
        if cluster_id == 0:  # Excitatory
            ei_balance += weight * 1
        elif cluster_id == 1:  # Inhibitory
            ei_balance += weight * (-1)
        elif cluster_id == 2:  # Other/Unknown
            ei_balance += weight * 0
    
    return ei_balance 

# Create mapping from nwb_paths index to actual insertion index
nwb_to_insertion_map = {}
insertion_counter = 0

for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    template = "*imec*"
    keys = [key for key in keys if fnmatch.fnmatch(key, template)]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys
    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
    # For NP137, remove 'imec1'
    if 'NP89' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        nwb_to_insertion_map[(i, s)] = insertion_counter
        insertion_counter += 1

print(f"NWB to insertion mapping created. Total insertions mapped: {insertion_counter}")

# Storage for results
assembly_results = []
neuron_results = []
insertion_results = []

# Process each insertion
for i in range(len(nwb_paths)):
    data = nap.load_file(nwb_paths[i])
    keys = data.keys()

    template = "*imec*"
    keys = [key for key in keys if fnmatch.fnmatch(key, template)]
    ks_keys = [key for key in keys if "KS4" in key]
    if ks_keys:
        keys = ks_keys
    th8_keys = [key for key in keys if "Th=8" in key]
    if th8_keys:
        keys = th8_keys
    else:
        th_keys = [key for key in keys if "Th=" in key]
        if th_keys:
            keys = th_keys
    template_sentgen = "*sentgen*"
    template_auto = "*_auto*"
    keys = [key for key in keys if not fnmatch.fnmatch(key, template_sentgen) and not fnmatch.fnmatch(key, template_auto)]
    # For NP137, remove 'imec1'
    if 'NP89' in str(nwb_paths[i]):
        keys = [key for key in keys if 'imec1' not in key]
    keys = sorted(keys, key=imec_key_sorter)

    for s in range(len(keys)):
        # Get the correct insertion index using our mapping
        insertion_idx = nwb_to_insertion_map[(i, s)]
        
        print(f"\nProcessing insertion {insertion_idx}: {keys[s]}")
        print(f"Region = {region_list[insertion_idx]}")
        print(f"Subject = {subj_list[insertion_idx]}")

        spike_times = data[keys[s]]

        # Get firing rates and task times
        firingRates_all = spike_times.metadata["rate"]
        if "TaskTimes" in data.keys():
            task_times = data["TaskTimes"]
            beh_epochs = nap.IntervalSet(start=task_times.start, end=task_times.end)
            spike_times_beh = spike_times.restrict(beh_epochs)
            firingRates_beh = spike_times_beh.metadata["rate"]
            start_time = task_times.start
            end_time = task_times.end
        else:
            spike_times_beh = spike_times
            firingRates_beh = firingRates_all
            all_spike_times = np.concatenate([spike_times[u].as_series().index.values for u in spike_times])
            if all_spike_times.size > 0:
                start_time = np.array([all_spike_times.min()])
                end_time = np.array([all_spike_times.max()])
            else:
                start_time = np.array([0.])
                end_time = np.array([0.])

        ks_stats = np.zeros(len(spike_times))
        ks_pvals = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            test = spike_times[u].as_series().index.values
            if len(test) > 1:
                min_time = start_time[0]
                max_time = end_time[-1]
                normalized_spike_times = (test - min_time) / (max_time - min_time)
                ks_result = kstest(normalized_spike_times, 'uniform')
                ks_stats[u] = ks_result.statistic
                ks_pvals[u] = ks_result.pvalue
            else:
                ks_stats[u] = np.nan
                ks_pvals[u] = np.nan

        violationThreshold = 3 / 1000
        violationPct = np.zeros(len(spike_times))
        for u in range(len(spike_times)):
            unit = spike_times[u]
            unit = unit.as_series().index
            if len(unit) < 100:
                violationPct[u] = 1
            else:
                isi = unit.diff()[1:]
                violations = np.where(isi < violationThreshold)
                violations = np.array(violations)
                violationPct[u] = violations.size / len(isi)

        if "KSLabel" in spike_times.metadata:
            KSLabels = spike_times.metadata["KSLabel"]
        else:
            KSLabels = spike_times.metadata["quality"]
        firingRates = firingRates_beh
        mask1 = violationPct < 3 / 100
        mask2 = firingRates > 0.5
        mask3 = KSLabels != "noise"
        mask4 = ks_stats < 0.3
        mask = mask1 & mask2 & mask3 & mask4
        indicesFinal = firingRates.index[mask]
        # now exclude neurons manually rejected
        indicesFinal = np.setdiff1d(indicesFinal, manual_exclude_lists[insertion_idx])
        spike_times_good = spike_times[indicesFinal]
        print(f"Number of good neurons: {len(spike_times_good)}")

        if len(spike_times_good) == 0:
            print("No good neurons found, skipping insertion")
            continue

        # Get cluster labels and metadata for this insertion's neurons
        # Use same logic but generalized for "all insertions" - try to find per insertion neuron-level cluster assignments and metadata
        # Here, we try to find which metadata/label array to use: names assumed generic (update code as appropriate)
        insertion_cluster_labels = []
        insertion_metadata = []

        # Find all global neurons for this insertion by insertion_idx
        insertion_neurons = []
        for global_idx, global_neuron in enumerate(comprehensive_metadata):  # rename as appropriate if you use a different neuron metadata array
            if (global_neuron['insertion_idx'] == insertion_idx):
                insertion_neurons.append((global_idx, global_neuron))
        
        print(f"Found {len(insertion_neurons)} neurons in global data for insertion {insertion_idx}")
        
        # Now match by sequential position, not by actual neuron index
        for local_neuron_idx, neuron_idx in enumerate(indicesFinal):
            # Use the sequential position (local_neuron_idx) to find the corresponding global neuron
            if local_neuron_idx < len(insertion_neurons):
                global_idx, global_neuron = insertion_neurons[local_neuron_idx]
                cluster_label = combined_comp_labels[global_idx]  # rename as appropriate
                insertion_cluster_labels.append(cluster_label)
                insertion_metadata.append(global_neuron)
            else:
                print(f"Warning: Not enough global neurons for local position {local_neuron_idx}")
                insertion_cluster_labels.append(-1)  # Unknown cluster
                insertion_metadata.append({})

        insertion_cluster_labels = np.array(insertion_cluster_labels)
        
        # Debug: Print cluster distribution for this insertion
        cluster_counts = [0, 0, 0]
        for label in insertion_cluster_labels:
            if 0 <= label <= 2:
                cluster_counts[label] += 1
        
        total_neurons = sum(cluster_counts)
        cluster_props = [count/total_neurons if total_neurons > 0 else 0 for count in cluster_counts]
        
        print(f"Cluster distribution for insertion {insertion_idx}:")
        print(f"  Cluster 0: {cluster_counts[0]} ({cluster_props[0]:.3f})")
        print(f"  Cluster 1: {cluster_counts[1]} ({cluster_props[1]:.3f})")
        print(f"  Cluster 2: {cluster_counts[2]} ({cluster_props[2]:.3f})")
        print(f"  Total: {total_neurons} neurons")

        timescale = 25 / 1000
        spikeCountMatrix = spike_times_good.count(bin_size=timescale)
        bin_edges = spikeCountMatrix.index.values
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        if len(bin_centers) < len(bin_edges):
            last_width = bin_edges[-1] - bin_edges[-2] if len(bin_edges) > 1 else 0
            last_center = bin_edges[-1] - last_width / 2
            bin_centers = np.append(bin_centers, last_center)
        spikeCountMatrix = spikeCountMatrix.values
        firingRateMatrix = spikeCountMatrix / timescale
        firingRateMatrix = stats.zscore(firingRateMatrix, axis=0)

        pca = PCA()
        firingRateMatrix_pca = pca.fit_transform(firingRateMatrix)
        eigenvalues = pca.explained_variance_
        upperbound = (1 + np.sqrt(firingRateMatrix.shape[1] / firingRateMatrix.shape[0])) ** 2
        assemblyIndices = np.where(eigenvalues > upperbound)
        assemblyEigenvalues = eigenvalues[assemblyIndices]
        print(f"Number of assembly patterns: {len(assemblyEigenvalues)}")

        n_pcs = len(assemblyEigenvalues)
        pc_vectors = pca.components_[:n_pcs, :]

        projections = firingRateMatrix @ pc_vectors.T

        if n_pcs > 0:
            fastica = FastICA(
                n_components=n_pcs,
                algorithm='parallel',
                whiten='unit-variance',
                max_iter=500,
                tol=1e-7,
                random_state=1
            )
            ica_components = fastica.fit_transform(projections)
            mixing_matrix = fastica.mixing_
            unmixing_matrix = fastica.components_
            ica_assembly_patterns = unmixing_matrix @ pc_vectors

            ap_norm_matrix = ica_assembly_patterns
            ap_norm_member = np.zeros(ica_assembly_patterns.shape)
            for ii in range(n_pcs):
                ap = ica_assembly_patterns[ii, :]
                ap_norm = ap / norm(ap)
                maxW = np.max(ap_norm)
                minW = np.min(ap_norm)
                if abs(minW) > maxW:
                    ap_norm = ap_norm * -1
                ap_norm_matrix[ii, :] = ap_norm
                threshold = np.mean(ap_norm) + np.std(ap_norm) * 2
                for c in range(len(ap)):
                    if ap_norm[c] > threshold:
                        ap_norm_member[ii, c] = 1

            sparsity_list = []
            for ap in range(len(ap_norm_matrix)):
                w = np.array(ap_norm_matrix[ap, :])
                n = w.size
                numerator = np.sqrt(n) - np.sum(np.abs(w))
                denominator = np.sqrt(n) - 1
                sparsity = 1 - numerator / denominator if denominator != 0 else np.nan
                sparsity_list.append(sparsity)

            # --- ASSEMBLY INFORMATION CAPACITY ANALYSIS ---
            insertion_assembly_results = []
            
            for assembly_idx in range(n_pcs):
                print(f"Analyzing assembly {assembly_idx + 1} of {n_pcs}")
                assembly_pattern = ap_norm_matrix[assembly_idx, :]
                assembly_members = ap_norm_member[assembly_idx, :]
                
                # Calculate E/I balance
                ei_balance = calculate_ei_balance(assembly_pattern, insertion_cluster_labels)
                
                # Calculate information capacity
                assembly_expression = ica_components[:, assembly_idx]
                assembly_info_capacity = calculate_information_capacity(assembly_expression, bin_centers)
                
                # Store detailed neuron information
                neuron_details = []
                for neuron_idx in range(len(assembly_pattern)):
                    # Handle case where metadata might be empty
                    if neuron_idx < len(insertion_metadata) and insertion_metadata[neuron_idx]:
                        depth = insertion_metadata[neuron_idx].get('depth', 0.0)
                        metadata = insertion_metadata[neuron_idx]
                    else:
                        depth = 0.0
                        metadata = {}
                    
                    neuron_details.append({
                        'neuron_idx': neuron_idx,
                        'assembly_weight': assembly_pattern[neuron_idx],
                        'is_member': bool(assembly_members[neuron_idx]),
                        'cluster_label': insertion_cluster_labels[neuron_idx],
                        'depth': depth,
                        'metadata': metadata
                    })
                
                assembly_result = {
                    'insertion_idx': insertion_idx,
                    'assembly_idx': assembly_idx,
                    'ei_balance': ei_balance,
                    'information_capacity': assembly_info_capacity,
                    'sparsity': sparsity_list[assembly_idx],
                    'n_neurons': len(assembly_pattern),
                    'n_members': np.sum(assembly_members),
                    'neuron_details': neuron_details
                }
                
                insertion_assembly_results.append(assembly_result)
                assembly_results.append(assembly_result)
            
            # --- NEURON INFORMATION CAPACITY ANALYSIS ---
            insertion_neuron_results = []
            
            for neuron_idx in range(firingRateMatrix.shape[1]):
                neuron_activity = firingRateMatrix[:, neuron_idx]
                neuron_info_capacity = calculate_information_capacity(neuron_activity, bin_centers)
                
                # Handle case where metadata might be empty
                if neuron_idx < len(insertion_metadata) and insertion_metadata[neuron_idx]:
                    depth = insertion_metadata[neuron_idx].get('depth', 0.0)
                    metadata = insertion_metadata[neuron_idx]
                else:
                    depth = 0.0
                    metadata = {}
                
                neuron_result = {
                    'insertion_idx': insertion_idx,
                    'neuron_idx': neuron_idx,
                    'cluster_label': insertion_cluster_labels[neuron_idx],
                    'depth': depth,
                    'information_capacity': neuron_info_capacity,
                    'metadata': metadata
                }
                
                insertion_neuron_results.append(neuron_result)
                neuron_results.append(neuron_result)
            
            insertion_result = {
                'insertion_idx': insertion_idx,
                'nwb_path': str(nwb_paths[i]),
                'session_key': keys[s],
                'region': region_list[insertion_idx],
                'subject': subj_list[insertion_idx],
                'pathology': path_list[insertion_idx],
                'grade': grade_list[insertion_idx],
                'n_neurons': len(spike_times_good),
                'n_assemblies': n_pcs,
                'assemblies': insertion_assembly_results,
                'neurons': insertion_neuron_results
            }
            
            insertion_results.append(insertion_result)
            
            print(f"Processed {n_pcs} assemblies and {len(spike_times_good)} neurons for insertion {insertion_idx}")
        else:
            print(f"No assemblies found for insertion {insertion_idx}")

print(f"\nAnalysis complete!")
print(f"Total insertions processed: {len(insertion_results)}")
print(f"Total assemblies analyzed: {len(assembly_results)}")
print(f"Total neurons analyzed: {len(neuron_results)}")

# Print summary
print(f"\n=== SUMMARY ===")
for result in insertion_results:
    print(f"Insertion {result['insertion_idx']}: {result['n_assemblies']} assemblies, "
          f"{result['n_neurons']} neurons, {result['region']}, {result['pathology']}-G{result['grade']}")

# Print E/I balance summary
print(f"\n=== E/I BALANCE SUMMARY ===")
ei_balances = [assembly['ei_balance'] for assembly in assembly_results]
info_capacities_assembly = [assembly['information_capacity']['capacity'] for assembly in assembly_results if not np.isnan(assembly['information_capacity']['capacity'])]

print(f"Assembly E/I Balance: mean = {np.mean(ei_balances):.3f}, std = {np.std(ei_balances):.3f}")
print(f"Assembly Information Capacity: mean = {np.mean(info_capacities_assembly):.3f}, std = {np.std(info_capacities_assembly):.3f}")

# Print neuron information capacity summary
print(f"\n=== NEURON INFORMATION CAPACITY SUMMARY ===")
info_capacities_neuron = [neuron['information_capacity']['capacity'] for neuron in neuron_results if not np.isnan(neuron['information_capacity']['capacity'])]

print(f"Neuron Information Capacity: mean = {np.mean(info_capacities_neuron):.3f}, std = {np.std(info_capacities_neuron):.3f}")

# Print cluster-specific neuron information capacity
print(f"\n=== NEURON INFORMATION CAPACITY BY CLUSTER ===")
for cluster_id in range(3):
    cluster_neurons = [neuron for neuron in neuron_results if neuron['cluster_label'] == cluster_id]
    cluster_capacities = [neuron['information_capacity']['capacity'] for neuron in cluster_neurons if not np.isnan(neuron['information_capacity']['capacity'])]
    
    if len(cluster_capacities) > 0:
        print(f"Cluster {cluster_id}: mean = {np.mean(cluster_capacities):.3f}, std = {np.std(cluster_capacities):.3f}, n = {len(cluster_capacities)}")
    else:
        print(f"Cluster {cluster_id}: no valid data")

In [ ]:
# REGRESSION-BASED YIELD CORRECTION FOR VALIDATION COHORT
# no correlation between yield and capacity in validation cohort

import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

print("="*80)
print("REGRESSION-BASED YIELD CORRECTION FOR VALIDATION COHORT")
print("="*80)

# Collect all capacity values and their corresponding yields
all_capacity_values = []
all_yield_values = []
assembly_indices = []  # Track which assembly each value belongs to

for i, assembly in enumerate(assembly_results):
    insertion_idx = assembly['insertion_idx']
    capacity_value = assembly['information_capacity']['capacity']
    
    # Skip NaN values
    if not np.isnan(capacity_value):
        all_capacity_values.append(capacity_value)
        all_yield_values.append(yield_list[insertion_idx])
        assembly_indices.append(i)

# Convert to numpy arrays
all_capacity_values = np.array(all_capacity_values)
all_yield_values = np.array(all_yield_values)
mean_yield = np.mean(all_yield_values)

print(f"\nData Summary:")
print(f"  Total assemblies: {len(all_capacity_values)}")
print(f"  Mean yield: {mean_yield:.2f}")
print(f"  Mean capacity: {np.mean(all_capacity_values):.4f} bits")
print(f"  Capacity range: [{np.min(all_capacity_values):.4f}, {np.max(all_capacity_values):.4f}] bits")

# Compute regression slope
slope, intercept = np.polyfit(all_yield_values, all_capacity_values, 1)
print(f"\n{'='*80}")
print("REGRESSION ANALYSIS")
print(f"{'='*80}")
print(f"Linear fit: capacity = {slope:.6f} * yield + {intercept:.6f}")
print(f"  Slope: {slope:.6f} bits/neuron")
print(f"  Intercept: {intercept:.6f} bits")

# Compute regression-based corrected capacity
regression_corrected_capacity = all_capacity_values - (slope * all_yield_values)

# Update the assembly_results in place with corrected values
for idx, assembly_idx in enumerate(assembly_indices):
    assembly_results[assembly_idx]['information_capacity']['capacity'] = regression_corrected_capacity[idx]

# Compare correlations
corr_orig, pval_orig = pearsonr(all_yield_values, all_capacity_values)
corr_regression, pval_regression = pearsonr(all_yield_values, regression_corrected_capacity)

print(f"\n{'='*80}")
print("CORRELATION ANALYSIS")
print(f"{'='*80}")
print(f"Original capacity vs yield:        r = {corr_orig:.6f}, p = {pval_orig:.2e}")
print(f"Regression-corrected vs yield:     r = {corr_regression:.6f}, p = {pval_regression:.2e}")
print(f"Correlation reduction: {((corr_orig - corr_regression) / corr_orig * 100):.2f}%")

print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"✅ Regression-based correction computed")
print(f"✅ Correlation with yield: {corr_regression:.6f} (target: 0.000)")
print(f"✅ Values updated in-place in assembly_results")
print(f"✅ Rest of notebook will use corrected values automatically")
print(f"{'='*80}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Regression-based Yield Correction', fontsize=16, fontweight='bold')

# Plot 1: Original
ax1 = axes[0]
ax1.scatter(all_yield_values, all_capacity_values, alpha=0.6, c='blue')
slope_plot, intercept_plot = np.polyfit(all_yield_values, all_capacity_values, 1)
x_fit = np.linspace(all_yield_values.min(), all_yield_values.max(), 100)
ax1.plot(x_fit, slope_plot * x_fit + intercept_plot, 'k--', lw=2, label=f'Fit: y={slope_plot:.4f}x+{intercept_plot:.2f}')
ax1.set_xlabel('Neuron Yield')
ax1.set_ylabel('Assembly Capacity (bits)')
ax1.set_title(f'Original: r={corr_orig:.4f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Corrected
ax2 = axes[1]
ax2.scatter(all_yield_values, regression_corrected_capacity, alpha=0.6, c='green')
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
# Add regression line for regression-corrected capacity
slope_reg, intercept_reg = np.polyfit(all_yield_values, regression_corrected_capacity, 1)
ax2.plot(x_fit, slope_reg * x_fit + intercept_reg, 'k--', lw=2, label=f'Fit: y={slope_reg:.4f}x+{intercept_reg:.2f}')
ax2.set_xlabel('Neuron Yield')
ax2.set_ylabel('Corrected Capacity')
ax2.set_title(f'Corrected: r={corr_regression:.4f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("regression_correction_validation_cohort.svg", dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete! Assembly capacity values have been corrected in-place.")
print("✅ All downstream analyses will automatically use the corrected values.")

In [ ]:
# ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION
# Focus on ratios + IDH mutation status + component information
plt.rcParams['svg.fonttype'] = 'none' 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for prettier plots
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['svg.fonttype'] = 'none'

# Create dataset - ALL GRADES
def create_enhanced_dataset():
    assembly_data = []
    for i, assembly in enumerate(assembly_results):
        insertion_idx = assembly['insertion_idx']
        
        # --------- REMOVED GRADE FILTER -------------
        # No grade filter; include all assemblies
        
        # Calculate tone metrics
        weights = []
        cluster_labels = []
        neuron_info_capacities = []
        
        for neuron_detail in assembly['neuron_details']:
            if neuron_detail['metadata']:
                weights.append(neuron_detail['assembly_weight'])
                cluster_labels.append(neuron_detail['cluster_label'])
                
                # Get neuron information capacity from neuron_results
                # Find the corresponding neuron in neuron_results
                neuron_idx = neuron_detail['neuron_idx']
                insertion_idx = assembly['insertion_idx']
                
                # Find the neuron in neuron_results
                neuron_info_capacity = 0.0
                for neuron_result in neuron_results:
                    if (neuron_result['insertion_idx'] == insertion_idx and 
                        neuron_result['neuron_idx'] == neuron_idx):
                        neuron_info_capacity = neuron_result['information_capacity']['capacity']
                        break
                
                neuron_info_capacities.append(neuron_info_capacity)
        
        weights = np.array(weights)
        cluster_labels = np.array(cluster_labels)
        neuron_info_capacities = np.array(neuron_info_capacities)
        
        excitatory_tone = np.sum(weights[cluster_labels == 2])
        inhibitory_tone = np.sum(weights[cluster_labels == 0])
        axonal_tone = np.sum(weights[cluster_labels == 1])
        
        # Calculate ratios
        total_tone = np.sum(np.abs(weights))
        excitatory_ratio = np.abs(excitatory_tone) / (total_tone + 1e-10)
        inhibitory_ratio = np.abs(inhibitory_tone) / (total_tone + 1e-10)
        axonal_ratio = np.abs(axonal_tone) / (total_tone + 1e-10)
        
        # Calculate component information (weighted sum of neuron information capacities)
        component_information = np.sum(weights * neuron_info_capacities)
        
        # Determine IDH mutation status - FIXED LABELING
        pathology = path_list[insertion_idx]
        idh_mutated = 1 if pathology in ['ast', 'oli'] else 0  # IDH-mut = 1, IDH-wt = 0
        
        # Create explicit IDH status labels for clarity
        idh_status = 'IDH-mut' if pathology in ['ast', 'oli'] else 'IDH-wt'
        
        assembly_data.append({
            'assembly_idx': i,
            'insertion_idx': insertion_idx,
            'pathology': pathology,
            'grade': grade_list[insertion_idx],
            'information_capacity': assembly['information_capacity']['capacity'],
            'excitatory_ratio': excitatory_ratio,
            'inhibitory_ratio': inhibitory_ratio,
            'axonal_ratio': axonal_ratio,
            'component_information': component_information,
            'idh_mutated': idh_mutated,
            'idh_status': idh_status  # Add explicit status label
        })
    
    return pd.DataFrame(assembly_data)

df = create_enhanced_dataset()
df_valid = df.dropna(subset=['information_capacity', 'component_information'])

print("=== ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION ===")
print("=== ALL GRADES INCLUDED ===")
print(f"Sample size: {len(df_valid)} assemblies")

# Check IDH distribution
idh_counts = df_valid['idh_mutated'].value_counts()
print(f"IDH-mutated (ast+oli): {idh_counts.get(1, 0)} assemblies")
print(f"IDH-wildtype (gbm): {idh_counts.get(0, 0)} assemblies")

# Verify IDH labeling
print(f"\n=== IDH LABELING VERIFICATION ===")
print("IDH Status breakdown:")
for status in df_valid['idh_status'].unique():
    subset = df_valid[df_valid['idh_status'] == status]
    print(f"  {status}: {len(subset)} assemblies, avg info capacity: {subset['information_capacity'].mean():.3f}")

# Define features (ratios + IDH mutation status + component information)
features = ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio', 'component_information', 'idh_mutated']
X = df_valid[features].values
y = df_valid['information_capacity'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 1. MULTIPLE REGRESSION MODEL
print(f"\n=== MULTIPLE REGRESSION MODEL ===")
lr = LinearRegression()
lr.fit(X_scaled, y)
y_pred = lr.predict(X_scaled)
r2 = lr.score(X_scaled, y)

print(f"R² = {r2:.3f}")
print(f"Coefficients:")
for i, feature in enumerate(features):
    print(f"  {feature}: {lr.coef_[i]:.3f}")

# 2. PERMUTATION TEST FOR MODEL SIGNIFICANCE
print(f"\n=== PERMUTATION TEST FOR MODEL SIGNIFICANCE ===")

def permutation_test_model(X, y, n_permutations=1000):
    """Test if model R² is significantly better than random"""
    
    # Original model
    lr_orig = LinearRegression()
    lr_orig.fit(X, y)
    r2_orig = lr_orig.score(X, y)
    
    # Permutation test
    r2_permuted = []
    for _ in range(n_permutations):
        y_perm = np.random.permutation(y)
        lr_perm = LinearRegression()
        lr_perm.fit(X, y_perm)
        r2_perm = lr_perm.score(X, y_perm)
        r2_permuted.append(r2_perm)
    
    r2_permuted = np.array(r2_permuted)
    p_value = np.mean(r2_permuted >= r2_orig)
    
    return r2_orig, r2_permuted, p_value

r2_orig, r2_permuted, p_model = permutation_test_model(X_scaled, y)

print(f"Original R² = {r2_orig:.3f}")
print(f"Permuted R² = {np.mean(r2_permuted):.3f} ± {np.std(r2_permuted):.3f}")
print(f"p-value = {p_model:.3f}")
print(f"Model is {'significant' if p_model < 0.05 else 'not significant'}")

# 3. MULTICOLLINEARITY ANALYSIS
print(f"\n=== MULTICOLLINEARITY ANALYSIS ===")

def calculate_vif(X, feature_names):
    """Calculate Variance Inflation Factor for each feature"""
    
    vif_scores = []
    for i in range(len(feature_names)):
        # Regress feature i against all other features
        X_other = np.delete(X, i, axis=1)
        y_feature = X[:, i]
        
        lr_vif = LinearRegression()
        lr_vif.fit(X_other, y_feature)
        r2_vif = lr_vif.score(X_other, y_feature)
        
        vif = 1 / (1 - r2_vif) if r2_vif < 0.99 else float('inf')
        vif_scores.append(vif)
        
        print(f"{feature_names[i]}: VIF = {vif:.2f}")
    
    return vif_scores

vif_scores = calculate_vif(X_scaled, features)

# 4. INDIVIDUAL CORRELATIONS
print(f"\n=== INDIVIDUAL CORRELATIONS ===")
individual_correlations = []
individual_p_values = []
for i, feature in enumerate(features):
    r, p = stats.pearsonr(y, X_scaled[:, i])
    individual_correlations.append(r)
    individual_p_values.append(p)
    print(f"{feature}: r = {r:.3f}, p = {p:.3f}")

# 5. COEFFICIENT SIGNIFICANCE TESTING
print(f"\n=== COEFFICIENT SIGNIFICANCE TESTING ===")

def test_coefficient_significance(X, y, feature_names, n_permutations=1000):
    """Test significance of individual coefficients using permutation"""
    
    # Original model
    lr_orig = LinearRegression()
    lr_orig.fit(X, y)
    coef_orig = lr_orig.coef_
    
    # Permutation test for each coefficient
    coef_permuted = np.zeros((n_permutations, len(feature_names)))
    
    for i in range(n_permutations):
        y_perm = np.random.permutation(y)
        lr_perm = LinearRegression()
        lr_perm.fit(X, y_perm)
        coef_permuted[i] = lr_perm.coef_
    
    # Calculate p-values
    p_values = []
    for i, feature in enumerate(feature_names):
        p_val = np.mean(np.abs(coef_permuted[:, i]) >= np.abs(coef_orig[i]))
        p_values.append(p_val)
        
        print(f"{feature}:")
        print(f"  Coefficient = {coef_orig[i]:.3f}")
        print(f"  p-value = {p_val:.3f}")
        print(f"  {'Significant' if p_val < 0.05 else 'Not significant'}")
        print()
    
    return coef_orig, p_values

coef_orig, coef_p_values = test_coefficient_significance(X_scaled, y, features)

# Convert to numpy arrays for comparison
coef_p_values = np.array(coef_p_values)
vif_scores = np.array(vif_scores)
individual_correlations = np.array(individual_correlations)
individual_p_values = np.array(individual_p_values)

# --- ONE-SIDED T-TESTS FOR GROUP COMPARISONS ---

print(f"\n=== IDH-SPECIFIC ANALYSIS ===")

# Compare information capacity between IDH groups - one-sided t-test
idh_mut = df_valid[df_valid['idh_mutated'] == 1]['information_capacity']
idh_wt = df_valid[df_valid['idh_mutated'] == 0]['information_capacity']

# We'll test IDH-mutated > IDH-wt (alternative='greater')
t_stat, t_p = stats.ttest_ind(idh_mut, idh_wt, alternative="greater")
print(f"Information Capacity by IDH Status:")
print(f"  IDH-mutated: {idh_mut.mean():.3f} ± {idh_mut.std():.3f} (n={len(idh_mut)})")
print(f"  IDH-wildtype: {idh_wt.mean():.3f} ± {idh_wt.std():.3f} (n={len(idh_wt)})")
print(f"  One-sided T-test (IDH-mutated > IDH-wt): t = {t_stat:.3f}, p = {t_p:.3f}")

# Compare ratios between IDH groups with one-sided t-tests
print(f"\nRatio Comparisons by IDH Status (one-sided, IDH-mutated > IDH-wt):")
idh_comparisons = {}
for ratio in ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio', 'component_information']:
    mut_vals = df_valid[df_valid['idh_mutated'] == 1][ratio]
    wt_vals = df_valid[df_valid['idh_mutated'] == 0][ratio]
    
    # one-sided: test (IDH-mutated > IDH-wt)
    t_stat_ratio, t_p_ratio = stats.ttest_ind(mut_vals, wt_vals, alternative="greater")
    idh_comparisons[ratio] = {
        't_stat': t_stat_ratio,
        'p_value': t_p_ratio,
        'mut_mean': mut_vals.mean(),
        'mut_std': mut_vals.std(),
        'wt_mean': wt_vals.mean(),
        'wt_std': wt_vals.std()
    }
    
    print(f"  {ratio}:")
    print(f"    IDH-mut: {mut_vals.mean():.3f} ± {mut_vals.std():.3f}")
    print(f"    IDH-wt: {wt_vals.mean():.3f} ± {wt_vals.std():.3f}")
    print(f"    One-sided T-test (IDH-mutated > IDH-wt): t = {t_stat_ratio:.3f}, p = {t_p_ratio:.3f}")

# Create comprehensive visualization with seaborn - FIXED LABELING
fig = plt.figure(figsize=(24, 20))
gs = fig.add_gridspec(5, 4, hspace=0.4, wspace=0.3)

fig.suptitle('ENHANCED SIGNIFICANCE TESTING WITH IDH MUTATION STATUS AND COMPONENT INFORMATION\nALL GRADES - Ratios + IDH Status + Component Information', 
             fontsize=16, fontweight='bold', y=0.95)

# 1. Correlation plots between ratios and information capacity - FIXED LABELING
ratio_features = ['excitatory_ratio', 'inhibitory_ratio', 'axonal_ratio']
for i, ratio in enumerate(ratio_features):
    ax = fig.add_subplot(gs[0, i])
    
    # Create scatter plot with explicit IDH status labels
    sns.scatterplot(data=df_valid, x=ratio, y='information_capacity', 
                   hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
                   alpha=0.7, ax=ax)
    
    # Add regression line
    sns.regplot(data=df_valid, x=ratio, y='information_capacity', 
                scatter=False, color='black', ax=ax)
    
    # Calculate correlation
    r, p = stats.pearsonr(df_valid[ratio], df_valid['information_capacity'])
    
    ax.set_title(f'{ratio.replace("_", " ").title()} vs Information Capacity\nr = {r:.3f}, p = {p:.3f}')
    ax.set_xlabel(ratio.replace('_', ' ').title())
    ax.set_ylabel('Information Capacity')
    ax.legend(title='IDH Status')

# 2. Component information vs information capacity - FIXED LABELING
ax = fig.add_subplot(gs[0, 3])
sns.scatterplot(data=df_valid, x='component_information', y='information_capacity', 
               hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
               alpha=0.7, ax=ax)
sns.regplot(data=df_valid, x='component_information', y='information_capacity', 
            scatter=False, color='black', ax=ax)

r, p = stats.pearsonr(df_valid['component_information'], df_valid['information_capacity'])
ax.set_title(f'Component Information vs Information Capacity\nr = {r:.3f}, p = {p:.3f}')
ax.set_xlabel('Component Information')
ax.set_ylabel('Information Capacity')
ax.legend(title='IDH Status')

# 3. Correlation plots between ratios - FIXED LABELING
ratio_pairs = [
    ('excitatory_ratio', 'inhibitory_ratio'),
    ('excitatory_ratio', 'axonal_ratio'),
    ('inhibitory_ratio', 'axonal_ratio')
]

for i, (ratio1, ratio2) in enumerate(ratio_pairs):
    ax = fig.add_subplot(gs[1, i])
    
    sns.scatterplot(data=df_valid, x=ratio1, y=ratio2, 
                   hue='idh_status', palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, 
                   alpha=0.7, ax=ax)
    sns.regplot(data=df_valid, x=ratio1, y=ratio2, 
                scatter=False, color='black', ax=ax)
    
    r, p = stats.pearsonr(df_valid[ratio1], df_valid[ratio2])
    ax.set_title(f'{ratio1.replace("_", " ").title()} vs {ratio2.replace("_", " ").title()}\nr = {r:.3f}, p = {p:.3f}')
    ax.set_xlabel(ratio1.replace('_', ' ').title())
    ax.set_ylabel(ratio2.replace('_', ' ').title())
    ax.legend(title='IDH Status')

# 4. Information capacity comparison by IDH status - FIXED LABELING
ax = fig.add_subplot(gs[1, 3])
sns.boxplot(data=df_valid, x='idh_status', y='information_capacity', 
            palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
sns.stripplot(data=df_valid, x='idh_status', y='information_capacity', 
              color='black', alpha=0.6, size=4, ax=ax)

# Add significance annotation
sig_text = f't = {t_stat:.3f}\np = {t_p:.3f}'
if t_p < 0.001:
    sig_text += '***'
elif t_p < 0.01:
    sig_text += '**'
elif t_p < 0.05:
    sig_text += '*'

ax.set_title(f'Information Capacity by IDH Status\n{sig_text}')
ax.set_xlabel('IDH Status')
ax.set_ylabel('Information Capacity')

# 5. Ratio comparisons by IDH status - FIXED LABELING
for i, ratio in enumerate(ratio_features):
    ax = fig.add_subplot(gs[2, i])
    sns.boxplot(data=df_valid, x='idh_status', y=ratio, 
                palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
    sns.stripplot(data=df_valid, x='idh_status', y=ratio, 
                  color='black', alpha=0.6, size=4, ax=ax)
    
    # Add significance annotation
    comp = idh_comparisons[ratio]
    sig_text = f't = {comp["t_stat"]:.3f}\np = {comp["p_value"]:.3f}'
    if comp["p_value"] < 0.001:
        sig_text += '***'
    elif comp["p_value"] < 0.01:
        sig_text += '**'
    elif comp["p_value"] < 0.05:
        sig_text += '*'
    
    ax.set_title(f'{ratio.replace("_", " ").title()} by IDH Status\n{sig_text}')
    ax.set_xlabel('IDH Status')
    ax.set_ylabel(ratio.replace('_', ' ').title())

# 6. Component information by IDH status - FIXED LABELING
ax = fig.add_subplot(gs[2, 3])
sns.boxplot(data=df_valid, x='idh_status', y='component_information', 
            palette={'IDH-wt': 'red', 'IDH-mut': 'blue'}, ax=ax)
sns.stripplot(data=df_valid, x='idh_status', y='component_information', 
              color='black', alpha=0.6, size=4, ax=ax)

# Add significance annotation
comp = idh_comparisons['component_information']
sig_text = f't = {comp["t_stat"]:.3f}\np = {comp["p_value"]:.3f}'
if comp["p_value"] < 0.001:
    sig_text += '***'
elif comp["p_value"] < 0.01:
    sig_text += '**'
elif comp["p_value"] < 0.05:
    sig_text += '*'

ax.set_title(f'Component Information by IDH Status\n{sig_text}')
ax.set_xlabel('IDH Status')
ax.set_ylabel('Component Information')

# 7. Model R² vs permutations
ax = fig.add_subplot(gs[3, 0])
sns.histplot(r2_permuted, bins=30, alpha=0.7, color='blue', ax=ax)
ax.axvline(r2_orig, color='red', linestyle='--', linewidth=2, label=f'Original R² = {r2_orig:.3f}')
ax.set_xlabel('R²')
ax.set_ylabel('Density')
ax.set_title('Model R² Permutation Test')
ax.legend()
ax.grid(True, alpha=0.3)

# 8. Coefficient significance
ax = fig.add_subplot(gs[3, 1])
colors = ['red' if p < 0.05 else 'blue' for p in coef_p_values]
bars = ax.bar(range(len(features)), coef_orig, color=colors, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('Coefficient Value')
ax.set_title('Coefficient Significance\n(Red = p < 0.05)')
ax.grid(True, alpha=0.3)

# 9. VIF scores
ax = fig.add_subplot(gs[3, 2])
colors_vif = ['red' if vif > 10 else 'orange' if vif > 5 else 'green' for vif in vif_scores]
bars = ax.bar(range(len(features)), vif_scores, color=colors_vif, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('VIF Score')
ax.set_title('Variance Inflation Factor\n(Red = High Multicollinearity)')
ax.axhline(y=10, color='red', linestyle='--', alpha=0.8, label='VIF = 10')
ax.axhline(y=5, color='orange', linestyle='--', alpha=0.8, label='VIF = 5')
ax.legend()
ax.grid(True, alpha=0.3)

# 10. Individual correlations
ax = fig.add_subplot(gs[3, 3])
colors_corr = ['red' if abs(r) > 0.5 else 'orange' if abs(r) > 0.3 else 'blue' for r in individual_correlations]
bars = ax.bar(range(len(features)), individual_correlations, color=colors_corr, alpha=0.7)
ax.set_xticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_ylabel('Correlation Coefficient')
ax.set_title('Individual Correlations\n(Red = Large Effect)')
ax.grid(True, alpha=0.3)

# 11. ENHANCED Feature correlation matrix with significance indicators
ax = fig.add_subplot(gs[4, :2])
corr_matrix = np.corrcoef(X_scaled.T)

# Create cleaner feature names for display
feature_names_display = ['Excitatory\nRatio', 'Inhibitory\nRatio', 'Axonal\nRatio', 'Component\nInformation', 'IDH\nMutated']

# Calculate significance for each correlation
sig_matrix = np.zeros_like(corr_matrix, dtype=bool)
for i in range(len(features)):
    for j in range(len(features)):
        if i != j:
            _, p_val = stats.pearsonr(X_scaled[:, i], X_scaled[:, j])
            if p_val < 0.05:
                sig_matrix[i, j] = True

# Plot correlation matrix with all values shown
im = sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
                 square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax,
                 xticklabels=feature_names_display, yticklabels=feature_names_display,
                 fmt='.2f', annot_kws={'fontsize': 10})

# Add significance indicators
for i in range(len(features)):
    for j in range(len(features)):
        if i != j and sig_matrix[i, j]:
            # Add asterisk for significant correlations
            ax.text(j + 0.5, i + 0.7, '*', ha='center', va='center', 
                   fontsize=14, fontweight='bold', color='white')

ax.set_title('Feature Correlation Matrix\n(* = p < 0.05)', fontsize=12, fontweight='bold')

# 12. Summary statistics
ax = fig.add_subplot(gs[4, 2:])
ax.text(0.05, 0.9, 'ENHANCED ANALYSIS SUMMARY (ALL GRADES)', fontsize=14, fontweight='bold')

# Model statistics
ax.text(0.05, 0.8, 'Model Performance:', fontsize=12, fontweight='bold')
ax.text(0.05, 0.75, f'• Sample size: {len(df_valid)} assemblies', fontsize=11)
ax.text(0.05, 0.7, f'• Model R²: {r2_orig:.3f}', fontsize=11)

# Fix the f-string issue by using a variable
significance_text = 'significant' if p_model < 0.05 else 'not significant'
ax.text(0.05, 0.65, f'• Model p-value: {p_model:.3f} ({significance_text})', fontsize=11)

# Feature analysis
ax.text(0.35, 0.8, 'Feature Analysis:', fontsize=12, fontweight='bold')
ax.text(0.35, 0.75, f'• {sum(coef_p_values < 0.05)}/{len(features)} coefficients significant', fontsize=11)
ax.text(0.35, 0.7, f'• {sum(vif_scores > 10)}/{len(features)} features have high VIF (>10)', fontsize=11)
ax.text(0.35, 0.65, f'• {sum(vif_scores > 5)}/{len(features)} features have moderate VIF (>5)', fontsize=11)

# Key findings
ax.text(0.65, 0.8, 'Key Findings:', fontsize=12, fontweight='bold')
for i, feature in enumerate(features):
    ax.text(0.65, 0.75 - i*0.05, f'• {feature}: r = {individual_correlations[i]:.3f}', fontsize=11)

# Component information interpretation
ax.text(0.05, 0.5, 'Component Information Interpretation:', fontsize=12, fontweight='bold')
ax.text(0.05, 0.45, '1. Weighted sum of neuron information capacities', fontsize=11)
ax.text(0.05, 0.4, '2. Assembly pattern × neuron information vector', fontsize=11)
ax.text(0.05, 0.35, '3. Captures information-weighted circuit organization', fontsize=11)
ax.text(0.05, 0.3, '4. Tests if information-rich neurons drive assemblies', fontsize=11)

# Recommendations
ax.text(0.35, 0.5, 'Recommendations:', fontsize=12, fontweight='bold')
ax.text(0.35, 0.45, '1. Compare component vs assembly information', fontsize=11)
ax.text(0.35, 0.4, '2. Test if information-rich neurons are key', fontsize=11)
ax.text(0.35, 0.35, '3. Validate weighted information hypothesis', fontsize=11)
ax.text(0.35, 0.3, '4. Focus on information-weighted circuit effects', fontsize=11)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.savefig("tone_model_fixed_labeling_validation.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n=== ENHANCED ANALYSIS SUMMARY (ALL GRADES) ===")
print("This analysis includes component information for all assemblies (all grades):")
print("1. Do ratios + component information predict capacity across grades?")
print("2. Is component information a key predictor regardless of tumor grade?")
print("3. Do information-rich neurons drive assemblies across disease spectrum?")
print("4. Test information-weighted circuit organization across glioma types")
print("5. Validate weighted information hypothesis in all-grade context")

In [ ]:
# STANDARD OLS COEFFICIENT SIGNIFICANCE TESTING
# Compare permutation test results with standard OLS statistics
print(f"\n{'='*80}")
print("STANDARD OLS COEFFICIENT SIGNIFICANCE TESTING")
print(f"{'='*80}")

import statsmodels.api as sm

# Add constant term for statsmodels (intercept)
X_with_const = sm.add_constant(X_scaled)

# Fit OLS model using statsmodels
ols_model = sm.OLS(y, X_with_const).fit()

# Get summary statistics
print(f"\nModel Summary:")
print(f"  R² = {ols_model.rsquared:.6f}")
print(f"  Adjusted R² = {ols_model.rsquared_adj:.6f}")
print(f"  F-statistic = {ols_model.fvalue:.4f}")
print(f"  F p-value = {ols_model.f_pvalue:.6f}")
print(f"  Degrees of freedom (model) = {ols_model.df_model:.0f}")
print(f"  Degrees of freedom (residual) = {ols_model.df_resid:.0f}")
print(f"  Total degrees of freedom = {ols_model.df_model + ols_model.df_resid:.0f}")

# Extract coefficient statistics (excluding intercept which is at index 0)
print(f"\n{'='*80}")
print("COEFFICIENT STATISTICS COMPARISON")
print(f"{'='*80}")
print(f"\n{'Feature':<25} {'Coefficient':<12} {'Std Error':<12} {'t-stat':<10} {'df':<6} {'p (OLS)':<12} {'p (Perm)':<12} {'Sig (OLS)':<10} {'Sig (Perm)':<10}")
print("-" * 120)

# Compare permutation and OLS results
for i, feature in enumerate(features):
    # OLS statistics (coefficient index is i+1 because intercept is at index 0)
    coef_ols = ols_model.params[i+1]
    std_err_ols = ols_model.bse[i+1]
    t_stat_ols = ols_model.tvalues[i+1]
    p_val_ols = ols_model.pvalues[i+1]
    df_ols = ols_model.df_resid
    
    # Permutation statistics
    coef_perm = coef_orig[i]
    p_val_perm = coef_p_values[i]
    
    # Significance markers
    sig_ols = '***' if p_val_ols < 0.001 else '**' if p_val_ols < 0.01 else '*' if p_val_ols < 0.05 else 'ns'
    sig_perm = '***' if p_val_perm < 0.001 else '**' if p_val_perm < 0.01 else '*' if p_val_perm < 0.05 else 'ns'
    
    print(f"{feature:<25} {coef_ols:>11.6f} {std_err_ols:>11.6f} {t_stat_ols:>9.4f} {df_ols:>5.0f} {p_val_ols:>11.6f} {p_val_perm:>11.6f} {sig_ols:>9} {sig_perm:>9}")

# Intercept statistics
print(f"\n{'Intercept':<25} {ols_model.params[0]:>11.6f} {ols_model.bse[0]:>11.6f} {ols_model.tvalues[0]:>9.4f} {ols_model.df_resid:>5.0f} {ols_model.pvalues[0]:>11.6f} {'N/A':>11} {'***' if ols_model.pvalues[0] < 0.001 else '**' if ols_model.pvalues[0] < 0.01 else '*' if ols_model.pvalues[0] < 0.05 else 'ns':>9} {'N/A':>9}")

print(f"\n{'='*80}")
print("NOTES:")
print(f"{'='*80}")
print("  - OLS p-values: Two-tailed t-test p-values from standard OLS regression")
print("  - Perm p-values: Permutation test p-values (one-tailed, based on |coef|)")
print("  - df: Degrees of freedom for t-test (residual df)")
print("  - Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print(f"\n  Model degrees of freedom: {ols_model.df_model:.0f}")
print(f"  Residual degrees of freedom: {ols_model.df_resid:.0f}")
print(f"  Total sample size: {len(y)}")
print(f"  Number of predictors: {len(features)}")


In [ ]:
# LOAD MODEL AND PREDICT ON VALIDATION COHORT
# Apply the trained Grade 4 model to the validation cohort assemblies

import joblib
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['svg.fonttype'] = 'none'
sns.set_style("whitegrid")

# Set a seed for reproducibility in permutation test
PERMUTATION_RANDOM_SEED = 42  # or any fixed int you choose

# Load the saved model
model_filename = '/userdata/gumbach/git_repos/tumor_np/revision/information_capacity_model_v2.pkl'
print("Loading model from:", model_filename)
model_package = joblib.load(model_filename)

print("=" * 80)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 80)
print(f"Model R² (training): {model_package['model_metadata']['r2_score']:.3f}")
print(f"Training sample size: {model_package['model_metadata']['n_samples']}")
print(f"Features: {model_package['feature_names']}")

def create_validation_dataset():
    """Extract features from validation cohort assemblies (same structure as training data)"""
    assembly_data = []
    for i, assembly in enumerate(assembly_results):
        insertion_idx = assembly['insertion_idx']
        weights = []
        cluster_labels = []
        neuron_info_capacities = []
        for neuron_detail in assembly['neuron_details']:
            if neuron_detail['metadata']:
                weights.append(neuron_detail['assembly_weight'])
                cluster_labels.append(neuron_detail['cluster_label'])
                neuron_idx = neuron_detail['neuron_idx']
                neuron_info_capacity = 0.0
                for neuron_result in neuron_results:
                    if (neuron_result['insertion_idx'] == insertion_idx and 
                        neuron_result['neuron_idx'] == neuron_idx):
                        neuron_info_capacity = neuron_result['information_capacity']['capacity']
                        break
                neuron_info_capacities.append(neuron_info_capacity)
        if len(weights) == 0:
            continue
        weights = np.array(weights)
        cluster_labels = np.array(cluster_labels)
        neuron_info_capacities = np.array(neuron_info_capacities)
        excitatory_tone = np.sum(weights[cluster_labels == 2])
        inhibitory_tone = np.sum(weights[cluster_labels == 0])
        axonal_tone = np.sum(weights[cluster_labels == 1])
        total_tone = np.sum(np.abs(weights))
        excitatory_ratio = np.abs(excitatory_tone) / (total_tone + 1e-10)
        inhibitory_ratio = np.abs(inhibitory_tone) / (total_tone + 1e-10)
        axonal_ratio = np.abs(axonal_tone) / (total_tone + 1e-10)
        component_information = np.sum(weights * neuron_info_capacities)
        pathology = path_list[insertion_idx]
        idh_mutated = 1 if pathology in ['ast', 'oli'] else 0
        assembly_data.append({
            'assembly_idx': i,
            'insertion_idx': insertion_idx,
            'pathology': pathology,
            'grade': grade_list[insertion_idx],
            'information_capacity': assembly['information_capacity']['capacity'],
            'excitatory_ratio': excitatory_ratio,
            'inhibitory_ratio': inhibitory_ratio,
            'axonal_ratio': axonal_ratio,
            'component_information': component_information,
            'idh_mutated': idh_mutated
        })
    return pd.DataFrame(assembly_data)

print("\n" + "=" * 80)
print("CREATING VALIDATION DATASET")
print("=" * 80)
df_validation = create_validation_dataset()
df_validation_valid = df_validation.dropna(subset=['information_capacity', 'component_information'])

print(f"Total assemblies in validation cohort: {len(df_validation)}")
print(f"Valid assemblies (no NaN): {len(df_validation_valid)}")

# Check grade distribution
print(f"\nGrade distribution:")
grade_counts = df_validation_valid['grade'].value_counts().sort_index()
for grade, count in grade_counts.items():
    print(f"  Grade {grade}: {count} assemblies")

# Extract features in the same order as training
feature_names = model_package['feature_names']
X_validation = df_validation_valid[feature_names].values
y_validation = df_validation_valid['information_capacity'].values

print(f"\nFeatures extracted (in order):")
for i, feat in enumerate(feature_names):
    print(f"  {i+1}. {feat}")

# Scale features using the SAME scaler from training
print("\nScaling features using trained scaler...")
X_validation_scaled = model_package['scaler'].transform(X_validation)

# Make predictions
print("Making predictions...")
y_predicted = model_package['model'].predict(X_validation_scaled)

# Calculate performance metrics
r2_validation = r2_score(y_validation, y_predicted)
rmse_validation = np.sqrt(mean_squared_error(y_validation, y_predicted))
mae_validation = mean_absolute_error(y_validation, y_predicted)
corr_coef, corr_p_value = stats.pearsonr(y_validation, y_predicted)

print("\n" + "=" * 80)
print("PREDICTION RESULTS ON VALIDATION COHORT")
print("=" * 80)
print(f"Sample size: {len(df_validation_valid)} assemblies")
print(f"\nPerformance Metrics:")
print(f"  R² = {r2_validation:.3f}")
print(f"  RMSE = {rmse_validation:.3f}")
print(f"  MAE = {mae_validation:.3f}")
print(f"\nCorrelation Analysis:")
print(f"  Pearson r = {corr_coef:.3f}")
print(f"  p-value = {corr_p_value:.6f}")
print(f"  {'Significant' if corr_p_value < 0.05 else 'Not significant'} (α = 0.05)")

# Permutation test for correlation (r)
print("\n" + "=" * 80)
print("SIGNIFICANCE TESTING FOR MODEL CORRELATION (Permutation)")
print("=" * 80)

def permutation_test_correlation(X, y_actual, model, scaler, n_permutations=1000, random_seed=None):
    """
    One-sided permutation test for correlation > 0.
    Calculates the p-value as the fraction of permuted correlations greater than or equal to the original correlation.
    """
    rng = np.random.default_rng(random_seed)  # use Generator for reproducibility
    X_scaled = scaler.transform(X)
    y_pred_orig = model.predict(X_scaled)
    corr_orig, _ = stats.pearsonr(y_actual, y_pred_orig)
    corr_permuted = []
    for _ in range(n_permutations):
        y_perm = rng.permutation(y_actual)
        corr_perm, _ = stats.pearsonr(y_perm, y_pred_orig)
        corr_permuted.append(corr_perm)
    corr_permuted = np.array(corr_permuted)
    # One-sided (greater): probability that a randomly permuted correlation >= observed
    p_value_corr = np.mean(corr_permuted >= corr_orig)
    return corr_orig, corr_permuted, p_value_corr

corr_orig_val, corr_permuted_val, p_value_corr_val = permutation_test_correlation(
    X_validation, y_validation, model_package['model'], model_package['scaler'],
    n_permutations=1000, random_seed=PERMUTATION_RANDOM_SEED
)

print(f"\nPermutation Test Results (n={1000} permutations):")
print(f"  Original correlation = {corr_orig_val:.3f}")
print(f"  Permuted correlation = {np.mean(corr_permuted_val):.3f} ± {np.std(corr_permuted_val):.3f}")
print(f"  One-sided p-value = {p_value_corr_val:.6f}")
print(f"  Correlation is {'SIGNIFICANT' if p_value_corr_val < 0.05 else 'NOT SIGNIFICANT'} (α = 0.05)")

# --- PLOT: (1) Model fit, (2) Permutation histogram for correlation only ---
# Now we want the correlation/fit plot (scatter) to be twice as wide as the histogram.

fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 3, width_ratios=[2, 1, 0.1], wspace=0.18)

# Scatter plot: predicted vs actual (left, twice as wide)
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_validation, y_predicted, alpha=0.7, color="purple", edgecolor="k")

# Correlation/predict line (add best linear fit)
slope, intercept = np.polyfit(y_validation, y_predicted, 1)
line_x = np.array([np.min(y_validation), np.max(y_validation)])
line_y = slope * line_x + intercept
ax1.plot(line_x, line_y, color="black", linestyle="-", linewidth=2, label="Predicted Trend")
# Ideal/reference line
ax1.plot(line_x, line_x, "r--", linewidth=1.5, label="Ideal")

# Annotate stats and significance in plot
sig_str = f"(p = {corr_p_value:.4f}{'*, sig.' if corr_p_value < 0.05 else ', n.s.'})"
ax1.text(0.05, 0.95, f"$R^2$ = {r2_validation:.3f}\nr = {corr_coef:.3f}\n{sig_str}", 
         fontsize=12, ha="left", va="top", transform=ax1.transAxes, bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray'))

ax1.set_xlabel("Observed Information Capacity", fontsize=12)
ax1.set_ylabel("Predicted Information Capacity", fontsize=12)
ax1.set_title(f"Model Fit: Validation Cohort", fontsize=13)
ax1.legend(loc="upper left", fontsize=10)
ax1.grid(alpha=0.3)

# Histogram: permutation test (right)
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(corr_permuted_val, bins=30, alpha=0.8, color='blue')
ax2.axvline(corr_orig_val, color='red', linestyle='--', linewidth=2, label=f'Observed r = {corr_orig_val:.3f}')
ax2.set_xlabel('Correlation Coefficient', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title(f'Permutation Test on r\none-sided p = {p_value_corr_val:.4f}', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("model_validation_results_permcor.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)
print(f"\nFigure saved as: model_validation_results_permcor.svg")

In [ ]:
# PAIRED YIELD PLOT: NEURON YIELD (LEFT) vs ASSEMBLY YIELD (RIGHT)
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['svg.fonttype'] = 'none' 

assembly_yield_list = [4, 3, 1, 5, 3, 3]

# Make sure yield_list and assembly_yield_list are defined and same length
x = np.array(yield_list)  # Neuron yield
y = np.array(assembly_yield_list)  # Assembly yield

# Get IDH mutation status for each insertion
# IDH-mut = ast or oli, IDH-wt = gbm
idh_status = []
for i in range(len(path_list)):
    if path_list[i] in ['ast', 'oli']:
        idh_status.append('IDH-mut')
    else:
        idh_status.append('IDH-wt')

idh_status = np.array(idh_status)

# Separate by IDH status for coloring
idh_mut_mask = idh_status == 'IDH-mut'
idh_wt_mask = idh_status == 'IDH-wt'

# X positions - reduce distance by a factor of 5 *again* (so 1/25th of original)
x_pos_neuron = 0
x_pos_assembly = 0.04  # original: 1.0, after one reduction: 0.2, after another: 0.04

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# Plot neuron yield points (left side) - colored by IDH status
if np.any(idh_mut_mask):
    ax.scatter(np.full(np.sum(idh_mut_mask), x_pos_neuron), x[idh_mut_mask], 
               color='tab:blue', s=100, alpha=0.7, label='Neuron Yield (IDH-mut)', 
               zorder=5, edgecolors='black', linewidth=0.5)
if np.any(idh_wt_mask):
    ax.scatter(np.full(np.sum(idh_wt_mask), x_pos_neuron), x[idh_wt_mask],
               color='lightblue', s=100, alpha=0.7, label='Neuron Yield (IDH-wt)',
               zorder=5, edgecolors='black', linewidth=0.5)

# Plot assembly yield points (right side) - colored by IDH status
if np.any(idh_mut_mask):
    ax.scatter(np.full(np.sum(idh_mut_mask), x_pos_assembly), y[idh_mut_mask],
               color='tab:orange', s=100, alpha=0.7, label='Assembly Yield (IDH-mut)',
               zorder=5, edgecolors='black', linewidth=0.5)
if np.any(idh_wt_mask):
    ax.scatter(np.full(np.sum(idh_wt_mask), x_pos_assembly), y[idh_wt_mask],
               color='moccasin', s=100, alpha=0.7, label='Assembly Yield (IDH-wt)',
               zorder=5, edgecolors='black', linewidth=0.5)

# Draw lines connecting paired values (matching indices)
# Color lines by IDH status
for i in range(len(x)):
    line_color = 'tab:blue' if idh_status[i] == 'IDH-mut' else 'lightblue'
    line_alpha = 0.5 if idh_status[i] == 'IDH-mut' else 0.3
    ax.plot([x_pos_neuron, x_pos_assembly], [x[i], y[i]], 
           color=line_color, alpha=line_alpha, linewidth=1.5, zorder=1)

# Set labels and formatting
ax.set_xticks([x_pos_neuron, x_pos_assembly])
ax.set_xticklabels(['Neuron Yield', 'Assembly Yield'], fontsize=12, fontweight='bold')
ax.set_ylabel('Yield Value (Count)', fontsize=12, fontweight='bold')
ax.set_title('Neuron Yield vs Assembly Yield\n(paired values connected by IDH status)', 
             fontsize=14, fontweight='bold')

# Set x limits, tightened to new spacing
ax.set_xlim(-0.6, 0.1)

# Legend
ax.legend(loc='upper right', frameon=True, fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig("yield_paired_neuron_vs_assembly.svg", format="svg", dpi=300, bbox_inches="tight")
plt.show()

# Print summary by IDH status
print("=" * 80)
print("YIELD SUMMARY BY IDH MUTATION STATUS")
print("=" * 80)

if np.any(idh_mut_mask):
    neuron_mut = x[idh_mut_mask]
    assembly_mut = y[idh_mut_mask]
    print(f"\nIDH-mut (n={np.sum(idh_mut_mask)}):")
    print(f"  Neuron Yield: mean={np.mean(neuron_mut):.2f}, std={np.std(neuron_mut):.2f}")
    print(f"  Assembly Yield: mean={np.mean(assembly_mut):.2f}, std={np.std(assembly_mut):.2f}")
    if len(neuron_mut) > 1:
        corr_mut = np.corrcoef(neuron_mut, assembly_mut)[0, 1]
        print(f"  Correlation: r = {corr_mut:.3f}")

if np.any(idh_wt_mask):
    neuron_wt = x[idh_wt_mask]
    assembly_wt = y[idh_wt_mask]
    print(f"\nIDH-wt (n={np.sum(idh_wt_mask)}):")
    print(f"  Neuron Yield: mean={np.mean(neuron_wt):.2f}, std={np.std(neuron_wt):.2f}")
    print(f"  Assembly Yield: mean={np.mean(assembly_wt):.2f}, std={np.std(assembly_wt):.2f}")
    if len(neuron_wt) > 1:
        corr_wt = np.corrcoef(neuron_wt, assembly_wt)[0, 1]
        print(f"  Correlation: r = {corr_wt:.3f}")

print(f"\nOverall (n={len(x)}):")
print(f"  Neuron Yield: mean={np.mean(x):.2f}, std={np.std(x):.2f}")
print(f"  Assembly Yield: mean={np.mean(y):.2f}, std={np.std(y):.2f}")
corr_overall = np.corrcoef(x, y)[0, 1]
print(f"  Correlation: r = {corr_overall:.3f}")
print("=" * 80)

In [ ]:
# CLUSTER PROPORTIONS COMPARISON BY IDH MUTATION STATUS (AST vs GBM)
plt.rcParams['svg.fonttype'] = 'none' 

import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

# Function to calculate proportions for pathology subgroups
def calculate_proportions_pathology(labels, metadata, pathology_values):
    """Calculate proportion of each cluster within a specific pathology group"""
    if isinstance(pathology_values, str):
        pathology_values = [pathology_values]
    
    group_mask = np.array([neuron['pathology'] in pathology_values for neuron in metadata])
    group_labels = labels[group_mask]
    
    if len(group_labels) == 0:
        return np.zeros(3)  # Return zeros if no data
    
    cluster_counts = np.bincount(group_labels, minlength=3)
    proportions = cluster_counts / np.sum(cluster_counts)
    return proportions

# Function to perform chi-square test for cluster proportions
def chi_square_test_clusters(labels, metadata, group1_values, group2_values):
    """Perform chi-square test comparing cluster distributions between two groups"""
    
    # Get masks for both groups
    group1_mask = np.array([neuron['pathology'] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron['pathology'] in group2_values for neuron in metadata])
    
    group1_labels = labels[group1_mask]
    group2_labels = labels[group2_mask]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None, None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    contingency_table = np.array([group1_counts, group2_counts])
    
    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    return chi2, p_value

# Function to perform post-hoc pairwise comparisons for individual clusters
def posthoc_cluster_comparisons(labels, metadata, group1_values, group2_values):
    """Perform post-hoc comparisons for each individual cluster"""
    
    # Get masks for both groups
    group1_mask = np.array([neuron['pathology'] in group1_values for neuron in metadata])
    group2_mask = np.array([neuron['pathology'] in group2_values for neuron in metadata])
    
    group1_labels = labels[group1_mask]
    group2_labels = labels[group2_mask]
    
    if len(group1_labels) == 0 or len(group2_labels) == 0:
        return None
    
    # Create contingency table
    group1_counts = np.bincount(group1_labels, minlength=3)
    group2_counts = np.bincount(group2_labels, minlength=3)
    
    results = []
    
    # Test each cluster individually (cluster vs all others)
    for cluster_id in range(3):
        # Create 2x2 contingency table: cluster vs all others
        cluster1_count = group1_counts[cluster_id]
        other1_count = np.sum(group1_counts) - cluster1_count
        
        cluster2_count = group2_counts[cluster_id]
        other2_count = np.sum(group2_counts) - cluster2_count
        
        contingency_2x2 = np.array([[cluster1_count, other1_count],
                                   [cluster2_count, other2_count]])
        
        # Perform chi-square test
        chi2, p_value, dof, expected = chi2_contingency(contingency_2x2)
        
        results.append({
            'cluster': cluster_id,
            'chi2': chi2,
            'p_value': p_value,
            'group1_prop': cluster1_count / np.sum(group1_counts) if np.sum(group1_counts) > 0 else 0,
            'group2_prop': cluster2_count / np.sum(group2_counts) if np.sum(group2_counts) > 0 else 0,
            'group1_count': cluster1_count,
            'group2_count': cluster2_count,
            'group1_total': np.sum(group1_counts),
            'group2_total': np.sum(group2_counts)
        })
    
    return results

# Calculate proportions
# IDH-mutated: ast and oli (since ast is typically IDH-mutated)
# IDH-wildtype: gbm
ast_props = calculate_proportions_pathology(combined_comp_labels, comprehensive_metadata, ['ast'])
gbm_props = calculate_proportions_pathology(combined_comp_labels, comprehensive_metadata, ['gbm'])

# Also check if 'oli' exists in the data
if any([neuron['pathology'] == 'oli' for neuron in comprehensive_metadata]):
    oli_props = calculate_proportions_pathology(combined_comp_labels, comprehensive_metadata, ['oli'])
    # Combine ast and oli for IDH-mutated group
    ast_oli_props = calculate_proportions_pathology(combined_comp_labels, comprehensive_metadata, ['ast', 'oli'])
    idh_mutated_props = ast_oli_props
    idh_mutated_label = 'ast+oli (IDH-mut)'
else:
    idh_mutated_props = ast_props
    idh_mutated_label = 'ast (IDH-mut)'

print("=" * 80)
print("CLUSTER PROPORTIONS BY IDH MUTATION STATUS")
print("=" * 80)
print(f"\n{idh_mutated_label} proportions: {idh_mutated_props}")
print(f"gbm (IDH-wt) proportions: {gbm_props}")

# Perform statistical tests
print("\n=== STATISTICAL TESTS ===")

# Overall comparison (ast+oli vs gbm)
if any([neuron['pathology'] == 'oli' for neuron in comprehensive_metadata]):
    chi2_overall, p_overall = chi_square_test_clusters(combined_comp_labels, comprehensive_metadata, 
                                                        ['ast', 'oli'], ['gbm'])
    group1_name = 'ast+oli (IDH-mut)'
else:
    chi2_overall, p_overall = chi_square_test_clusters(combined_comp_labels, comprehensive_metadata, 
                                                        ['ast'], ['gbm'])
    group1_name = 'ast (IDH-mut)'

print(f"Overall comparison ({group1_name} vs gbm (IDH-wt)):")
if chi2_overall is not None:
    print(f"  Chi-square = {chi2_overall:.4f}, p-value = {p_overall:.4f}")

# Post-hoc pairwise comparisons
if any([neuron['pathology'] == 'oli' for neuron in comprehensive_metadata]):
    posthoc_results = posthoc_cluster_comparisons(combined_comp_labels, comprehensive_metadata, 
                                                  ['ast', 'oli'], ['gbm'])
else:
    posthoc_results = posthoc_cluster_comparisons(combined_comp_labels, comprehensive_metadata, 
                                                  ['ast'], ['gbm'])

if posthoc_results:
    print("\nPost-hoc pairwise comparisons (each cluster vs all others):")
    for result in posthoc_results:
        significance = "***" if result['p_value'] < 0.001 else "**" if result['p_value'] < 0.01 else "*" if result['p_value'] < 0.05 else "ns"
        print(f"  Cluster {result['cluster']}: χ² = {result['chi2']:.3f}, p = {result['p_value']:.4f} {significance}")
        print(f"    {group1_name}: prop = {result['group1_prop']:.3f} (n = {result['group1_count']}/{result['group1_total']})")
        print(f"    gbm (IDH-wt): prop = {result['group2_prop']:.3f} (n = {result['group2_count']}/{result['group2_total']})")

# Define colors for clusters
cluster_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
cluster_names = ['Cluster 0', 'Cluster 1', 'Cluster 2']

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('Cluster Proportions by IDH Mutation Status', fontsize=16, fontweight='bold')

# Prepare data
groups = [idh_mutated_label, 'gbm (IDH-wt)']
proportions_data = [idh_mutated_props, gbm_props]

# Create stacked bars
x_pos = np.arange(len(groups))
width = 0.6

bottom = np.zeros(len(groups))
for i, (cluster_props, color, name) in enumerate(zip(np.array(proportions_data).T, cluster_colors, cluster_names)):
    ax.bar(x_pos, cluster_props, width, bottom=bottom, color=color, alpha=0.8, 
            label=name, edgecolor='black', linewidth=1)
    # Add value labels in the middle of each segment
    for j, prop in enumerate(cluster_props):
        if prop > 0.05:  # Only label if proportion is > 5%
            ax.text(x_pos[j], bottom[j] + prop/2, f'{prop:.2f}', 
                   ha='center', va='center', fontweight='bold', fontsize=11, color='white')
    bottom += cluster_props

ax.set_ylabel('Proportion', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_xticks(x_pos)
ax.set_xticklabels(groups, fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels at the bottom showing all proportions
for i, group in enumerate(groups):
    cluster_props = proportions_data[i]
    label_text = f"({cluster_props[0]:.2f}, {cluster_props[1]:.2f}, {cluster_props[2]:.2f})"
    ax.text(i, -0.08, label_text, ha='center', va='top', fontweight='bold', fontsize=10)

# Add significance annotation
if chi2_overall is not None:
    significance = "***" if p_overall < 0.001 else "**" if p_overall < 0.01 else "*" if p_overall < 0.05 else "ns"
    ax.text(0.5, 1.02, f"Overall χ² = {chi2_overall:.4f}, p = {p_overall:.4f} {significance}", 
            ha='center', va='bottom', fontweight='bold', fontsize=12, 
            transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig("cluster_proportions_idh_status.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

# Print detailed cluster-by-cluster comparisons
print("\n=== DETAILED CLUSTER COMPARISONS ===")
print(f"Cluster proportions by IDH mutation status:")
for i in range(3):
    idh_mut_prop = idh_mutated_props[i]
    gbm_prop = gbm_props[i]
    diff = gbm_prop - idh_mut_prop
    print(f"  Cluster {i}:")
    print(f"    {group1_name}: {idh_mut_prop:.3f}")
    print(f"    gbm (IDH-wt): {gbm_prop:.3f}")
    print(f"    Difference: {diff:+.3f}")

print("\n" + "=" * 80)

what is chi(df = 2) = 17.4944 p value? 

In [ ]:
# HEATMAP SUMMARY OF ALL LISTS - VALIDATION COHORT
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams['svg.fonttype'] = 'none'

# All lists to visualize
all_lists = [
    subj_list,
    grade_list,
    path_list,
    yield_list,
    opercular_list,
    region_list,
    age_list,
    gender_list
]
list_names = [
    'subj',
    'grade',
    'path',
    'yield',
    'opercular',
    'region',
    'age',
    'gender'
]

# Prepare matrix for heatmap (numeric codes for categorical, numbers for numeric)
matrix = []
row_vmin = []
row_vmax = []
for idx, lst in enumerate(all_lists):
    # Special handling for region row
    if list_names[idx] == 'region':
        # pSTG = 2, parsOp and parsTr and vPrCG = 1, all others = 0
        region_numeric = []
        for r in lst:
            if r == 'pSTG':
                region_numeric.append(2)
            elif r == 'parsOp' or r == 'parsTr' or r == 'vPrCG':
                region_numeric.append(1)
            else:
                region_numeric.append(0)
        matrix.append(region_numeric)
        row_vmin.append(0)
        row_vmax.append(2)
    # Special handling for path row
    elif list_names[idx] == 'path':
        # oligo = 0, astro = 1, gbm = 2
        path_numeric = []
        for p in lst:
            if p.lower().startswith('oli'):
                path_numeric.append(0)
            elif p.lower().startswith('ast'):
                path_numeric.append(1)
            elif p.lower().startswith('gbm'):
                path_numeric.append(2)
            else:
                path_numeric.append(np.nan)  # fallback for unknown
        matrix.append(path_numeric)
        row_vmin.append(0)
        row_vmax.append(2)
    elif all(isinstance(x, (int, float, np.integer, np.floating)) for x in lst):
        matrix.append(lst)
        row_vmin.append(np.min(lst))
        row_vmax.append(np.max(lst))
    else:
        cat = pd.Categorical(lst)
        matrix.append(cat.codes)
        row_vmin.append(np.min(cat.codes))
        row_vmax.append(np.max(cat.codes))

matrix = np.array(matrix)

# Create a custom colormap from white to black
wb_cmap = LinearSegmentedColormap.from_list("wb", ["white", "black"])

# --- Make the heatmap and the entire plot with aspect ratio 3:1, with a single box around all cells (no grid) ---

n_rows, n_cols = matrix.shape
aspect_ratio = 0.3  # pbaspect 3:1 (width:height)
base = 4  # base size for the shorter side
if n_cols >= n_rows:
    figsize = (base * aspect_ratio, base)
else:
    figsize = (base, base * aspect_ratio)

fig, ax = plt.subplots(figsize=figsize)

# For each row, plot as a single-row heatmap (no colorbars here)
for i in range(n_rows):
    row = matrix[i][np.newaxis, :]
    ax.imshow(
        row,
        aspect='auto',
        cmap=wb_cmap,
        vmin=row_vmin[i],
        vmax=row_vmax[i],
        extent=[-0.5, n_cols-0.5, i+0.5, i-0.5]
    )

# For just the yield row, plot the number in the cell
yield_row_idx = list_names.index('yield')
for j in range(n_cols):
    val = all_lists[yield_row_idx][j]
    # Choose text color for contrast
    norm_val = (val - row_vmin[yield_row_idx]) / (row_vmax[yield_row_idx] - row_vmin[yield_row_idx] + 1e-8)
    text_color = 'white' if norm_val > 0.5 else 'black'
    ax.text(j, yield_row_idx, str(val), ha='center', va='center', fontsize=8, color=text_color, fontweight='bold')

# Set yticks to row names
ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels(list_names, fontsize=10)
ax.set_xticks(np.arange(n_cols))
ax.set_xticklabels([str(idx) for idx in range(n_cols)], rotation=90, fontsize=8)
ax.set_xlabel('Index in list (subject/sample)')

# Turn the grid off
ax.grid(False)

# Remove all spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Draw a single box around all cells using Rectangle from matplotlib.patches
rect = mpatches.Rectangle(
    (-0.5, -0.5),
    n_cols,
    n_rows,
    fill=False,
    edgecolor='black',
    linewidth=2
)
ax.add_patch(rect)

# Remove whitespace between rows
ax.set_ylim(n_rows-0.5, -0.5)
ax.set_xlim(-0.5, n_cols-0.5)

# Set the aspect ratio of the data region to 3:1 (width:height)
ax.set_aspect(aspect_ratio, adjustable='box')

plt.subplots_adjust(left=0.15, right=0.98, top=0.92, bottom=0.15)
plt.title('Validation Cohort: Stacked Heatmap of All Lists (white=low, black=high)', y=1.02)
plt.gcf().set_size_inches(figsize)
plt.savefig('heatmap_summary_validation.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

# --- Separate plot for colorbars ---
from matplotlib import gridspec

n_rows = len(all_lists)
fig_cb, axes_cb = plt.subplots(nrows=n_rows, ncols=1, figsize=(2, n_rows * 0.7), constrained_layout=True)

if n_rows == 1:
    axes_cb = [axes_cb]

for i in range(n_rows):
    # Create a dummy image for colorbar
    dummy = np.linspace(row_vmin[i], row_vmax[i], 256).reshape(1, -1)
    im = axes_cb[i].imshow(dummy, aspect='auto', cmap=wb_cmap, vmin=row_vmin[i], vmax=row_vmax[i])
    cbar = plt.colorbar(im, ax=axes_cb[i], orientation='horizontal', fraction=0.7, pad=0.2)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label(list_names[i], fontsize=8)
    axes_cb[i].axis('off')

fig_cb.suptitle('Colorbars for Each Row', fontsize=12, y=1.02)
plt.savefig('heatmap_colorbars_validation.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PIE CHART: PERCENTAGE OF NEURONS BY IDH MUTATION STATUS
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['svg.fonttype'] = 'none' 

# Count neurons by IDH mutation status
idh_mut_count = 0
idh_wt_count = 0
idh_mut_pathologies = []
idh_wt_pathologies = []

for neuron in comprehensive_metadata:
    pathology = neuron['pathology']
    if pathology in ['ast', 'oli']:
        idh_mut_count += 1
        if pathology not in idh_mut_pathologies:
            idh_mut_pathologies.append(pathology)
    elif pathology == 'gbm':
        idh_wt_count += 1
        if pathology not in idh_wt_pathologies:
            idh_wt_pathologies.append(pathology)

total_neurons = idh_mut_count + idh_wt_count

# Calculate percentages
idh_mut_pct = (idh_mut_count / total_neurons) * 100
idh_wt_pct = (idh_wt_count / total_neurons) * 100

print("=" * 80)
print("NEURON COUNT BY IDH MUTATION STATUS")
print("=" * 80)
print(f"\nTotal neurons: {total_neurons}")
print(f"\nIDH-mutated (ast + oli): {idh_mut_count} ({idh_mut_pct:.1f}%)")
if idh_mut_pathologies:
    print(f"  Pathologies included: {', '.join(idh_mut_pathologies)}")
print(f"\nIDH-wildtype (gbm): {idh_wt_count} ({idh_wt_pct:.1f}%)")

# Create simple pie chart
fig, ax = plt.subplots(figsize=(8, 8))

# Labels and values
labels = ['IDH-mutated', 'IDH-wildtype']
sizes = [idh_mut_count, idh_wt_count]
colors = ['#4CAF50', '#FF5722']  # Green for IDH-mut, Red for IDH-wt

# Create pie chart with percentage labels
wedges, texts, autotexts = ax.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)

# Customize percentage text
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')

ax.set_title(f'Neuron Distribution by IDH Mutation Status\nValidation Cohort (n={total_neurons} neurons)', 
             fontsize=16, fontweight='bold', pad=20)

# Equal aspect ratio ensures that pie is drawn as a circle
ax.axis('equal')

plt.tight_layout()
plt.savefig("neuron_idh_status_pie_chart.svg", format="svg", dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 80)
print("Figure saved as: neuron_idh_status_pie_chart.svg")
print("=" * 80)